In [3]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 0 — FROZEN R1/R2 DEPENDENCY + UNION CONTRACT VERIFICATION
# ==============================================================================

from pathlib import Path
import json
import hashlib
import platform
import sys

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 0 — FROZEN R1/R2 DEPENDENCY + UNION CONTRACT VERIFICATION")
print("=" * 80)


# ==============================================================================
# 1. ENVIRONMENT
# ==============================================================================

print("\n" + "=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python  :", sys.version)
print("OS      :", platform.platform())
print("pandas  :", pd.__version__)
print("pyarrow :", pa.__version__)


# ==============================================================================
# 2. PROJECT PATHS
# ==============================================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

R3_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R3_union"
)

R3_FROZEN_ROOT = (
    R3_ROOT
    / "frozen"
)

R3_DIAGNOSTIC_ROOT = (
    R3_ROOT
    / "diagnostics"
)


# ------------------------------------------------------------------------------
# Frozen R1
# ------------------------------------------------------------------------------

R1_FROZEN_ROOT = (
    R1_ROOT
    / "frozen"
)

R1_CANDIDATES = (
    R1_FROZEN_ROOT
    / "r1_sparse_candidates.parquet"
)

R1_MANIFEST = (
    R1_FROZEN_ROOT
    / "r1_cell6_freeze_manifest.json"
)


# ------------------------------------------------------------------------------
# Frozen R2
# ------------------------------------------------------------------------------

R2_FROZEN_ROOT = (
    R2_ROOT
    / "frozen"
)

R2_CANDIDATES = (
    R2_FROZEN_ROOT
    / "r2_dense_candidates.parquet"
)

R2_MANIFEST = (
    R2_FROZEN_ROOT
    / "r2_dense_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("R3 PATHS")
print("=" * 80)

print("R1 candidates :", R1_CANDIDATES)
print("R1 manifest   :", R1_MANIFEST)

print("R2 candidates :", R2_CANDIDATES)
print("R2 manifest   :", R2_MANIFEST)

print("R3 root       :", R3_ROOT)


# ==============================================================================
# 3. CREATE R3 DIRECTORIES
# ==============================================================================

R3_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R3_FROZEN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R3_DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 4. FROZEN ARTIFACT EXISTENCE
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN ARTIFACT EXISTENCE")
print("=" * 80)

assert R1_CANDIDATES.exists(), (
    f"Missing frozen R1 candidate artifact:\n{R1_CANDIDATES}"
)

assert R1_MANIFEST.exists(), (
    f"Missing R1 freeze manifest:\n{R1_MANIFEST}"
)

assert R2_CANDIDATES.exists(), (
    f"Missing frozen R2 candidate artifact:\n{R2_CANDIDATES}"
)

assert R2_MANIFEST.exists(), (
    f"Missing R2 freeze manifest:\n{R2_MANIFEST}"
)

print("R1 candidates :", True)
print("R1 manifest   :", True)
print("R2 candidates :", True)
print("R2 manifest   :", True)


# ==============================================================================
# 5. LOAD + VERIFY MANIFESTS
# ==============================================================================

with open(
    R1_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    r1_manifest = json.load(handle)


with open(
    R2_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    r2_manifest = json.load(handle)


assert r1_manifest["status"] == "FROZEN", (
    "R1 manifest is not FROZEN."
)

assert r2_manifest["status"] == "FROZEN", (
    "R2 manifest is not FROZEN."
)


print("\n" + "=" * 80)
print("FREEZE MANIFEST STATUS")
print("=" * 80)

print(
    "R1 status:",
    r1_manifest["status"],
)

print(
    "R2 status:",
    r2_manifest["status"],
)


# ==============================================================================
# 6. EXPECTED POPULATION CONTRACT
# ==============================================================================

EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_TOP_K = 50
EXPECTED_FOLDS = [0, 1, 2, 3, 4]


assert int(r1_manifest["responses"]) == EXPECTED_RESPONSES
assert int(r2_manifest["responses"]) == EXPECTED_RESPONSES

assert int(r1_manifest["sessions"]) == EXPECTED_SESSIONS
assert int(r2_manifest["sessions"]) == EXPECTED_SESSIONS

assert int(r1_manifest["top_k"]) == EXPECTED_TOP_K
assert int(r2_manifest["top_k"]) == EXPECTED_TOP_K


print("\n" + "=" * 80)
print("POPULATION CONTRACT")
print("=" * 80)

print(
    "Expected responses:",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "R1 responses:",
    f"{int(r1_manifest['responses']):,}",
)

print(
    "R2 responses:",
    f"{int(r2_manifest['responses']):,}",
)

print(
    "Expected sessions:",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "R1 sessions:",
    f"{int(r1_manifest['sessions']):,}",
)

print(
    "R2 sessions:",
    f"{int(r2_manifest['sessions']):,}",
)

print(
    "Expected Top-K:",
    EXPECTED_TOP_K,
)

print(
    "R1 Top-K:",
    int(r1_manifest["top_k"]),
)

print(
    "R2 Top-K:",
    int(r2_manifest["top_k"]),
)


# ==============================================================================
# 7. EXACT FROZEN SCHEMA CONTRACT
# ==============================================================================

R1_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

R2_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 8. PARQUET SCHEMA CHECK — NO FULL DATAFRAME LOAD
# ==============================================================================

r1_schema = pq.read_schema(
    R1_CANDIDATES
)

r2_schema = pq.read_schema(
    R2_CANDIDATES
)

r1_columns = r1_schema.names
r2_columns = r2_schema.names


assert r1_columns == R1_COLUMNS, (
    "R1 frozen schema mismatch.\n"
    f"Expected: {R1_COLUMNS}\n"
    f"Observed: {r1_columns}"
)

assert r2_columns == R2_COLUMNS, (
    "R2 frozen schema mismatch.\n"
    f"Expected: {R2_COLUMNS}\n"
    f"Observed: {r2_columns}"
)


print("\n" + "=" * 80)
print("FROZEN SCHEMA CONTRACT")
print("=" * 80)

print(
    "R1 schema:",
    "PASS",
)

print(
    "R2 schema:",
    "PASS",
)


# ==============================================================================
# 9. MANIFEST / PARQUET SCHEMA CONTRACT
# ==============================================================================

# R1 manifest contains an explicit column contract.
if "columns" in r1_manifest:

    assert (
        r1_manifest["columns"]
        == R1_COLUMNS
    ), (
        "R1 manifest column contract mismatch.\n"
        f"Expected: {R1_COLUMNS}\n"
        f"Observed: {r1_manifest['columns']}"
    )

    r1_manifest_schema_check = True

else:

    r1_manifest_schema_check = False


# R2 manifest does NOT necessarily contain a `columns` field.
# Therefore the authoritative schema check for R2 is the actual
# frozen Parquet schema already verified above.

if "columns" in r2_manifest:

    assert (
        r2_manifest["columns"]
        == R2_COLUMNS
    ), (
        "R2 manifest column contract mismatch.\n"
        f"Expected: {R2_COLUMNS}\n"
        f"Observed: {r2_manifest['columns']}"
    )

    r2_manifest_schema_check = True

else:

    r2_manifest_schema_check = False


print("\n" + "=" * 80)
print("MANIFEST / PARQUET SCHEMA CONTRACT")
print("=" * 80)

print(
    "R1 manifest explicit schema:",
    r1_manifest_schema_check,
)

print(
    "R2 manifest explicit schema:",
    r2_manifest_schema_check,
)

print(
    "R1 actual Parquet schema:",
    r1_columns == R1_COLUMNS,
)

print(
    "R2 actual Parquet schema:",
    r2_columns == R2_COLUMNS,
)

# Actual Parquet schema is the hard requirement.
assert r1_columns == R1_COLUMNS
assert r2_columns == R2_COLUMNS

print(
    "Schema contract:",
    "PASS",
)


# ==============================================================================
# 10. TARGET / LABEL ISOLATION
# ==============================================================================

PROHIBITED_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
    "prediction",
}

assert not (
    PROHIBITED_COLUMNS
    &
    set(r1_columns)
), (
    "R1 contains prohibited target/prediction columns."
)

assert not (
    PROHIBITED_COLUMNS
    &
    set(r2_columns)
), (
    "R2 contains prohibited target/prediction columns."
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print("R1 target leakage columns:", "NONE")
print("R2 target leakage columns:", "NONE")


# ==============================================================================
# 11. SHARED UNION KEY CONTRACT
# ==============================================================================

UNION_KEYS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
]

for column in UNION_KEYS:

    assert column in r1_columns, (
        f"R1 missing union key: {column}"
    )

    assert column in r2_columns, (
        f"R2 missing union key: {column}"
    )


print("\n" + "=" * 80)
print("UNION KEY CONTRACT")
print("=" * 80)

print(
    "Union keys:",
    UNION_KEYS,
)

print(
    "R1/R2 shared key contract:",
    "PASS",
)


# ==============================================================================
# 12. SAME-SESSION REQUIREMENT
# ==============================================================================

# R3 MUST NEVER create cross-session candidates.
#
# This is checked again after loading candidate rows in the next cell.
# Here we only establish that both frozen artifacts contain session_id.


assert "session_id" in r1_columns
assert "session_id" in r2_columns

print(
    "Same-session key available:",
    True,
)


# ==============================================================================
# 13. FOLD CONTRACT
# ==============================================================================
#
# IMPORTANT:
# R2 freeze manifest does NOT declare a `folds` field.
#
# Therefore the authoritative fold contract for R3 is the actual frozen
# candidate parquet, not an assumed manifest key.
#
# We only read the `fold` column here — we do NOT reload the full candidate
# tables into memory.
# ==============================================================================

EXPECTED_FOLDS = [0, 1, 2, 3, 4]


r1_fold_values = sorted(
    pd.read_parquet(
        R1_CANDIDATES,
        columns=["fold"],
        engine="pyarrow",
    )[
        "fold"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

r2_fold_values = sorted(
    pd.read_parquet(
        R2_CANDIDATES,
        columns=["fold"],
        engine="pyarrow",
    )[
        "fold"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


assert (
    r1_fold_values
    == EXPECTED_FOLDS
), (
    "Unexpected R1 fold values.\n"
    f"Expected: {EXPECTED_FOLDS}\n"
    f"Observed: {r1_fold_values}"
)


assert (
    r2_fold_values
    == EXPECTED_FOLDS
), (
    "Unexpected R2 fold values.\n"
    f"Expected: {EXPECTED_FOLDS}\n"
    f"Observed: {r2_fold_values}"
)


print("\n" + "=" * 80)
print("FOLD CONTRACT")
print("=" * 80)

print(
    "Expected folds:",
    EXPECTED_FOLDS,
)

print(
    "R1 frozen candidate folds:",
    r1_fold_values,
)

print(
    "R2 frozen candidate folds:",
    r2_fold_values,
)

print(
    "R1 fold contract:",
    "PASS",
)

print(
    "R2 fold contract:",
    "PASS",
)


# ==============================================================================
# 14. FINAL CELL GATE
# ==============================================================================

R3_CELL_0_READY = True

print("\n" + "=" * 80)
print("R3 CELL 0 STATUS")
print("=" * 80)

print(
    "R1 frozen dependency :",
    True,
)

print(
    "R2 frozen dependency :",
    True,
)

print(
    "Schema contract      :",
    True,
)

print(
    "Population contract  :",
    True,
)

print(
    "Target isolation     :",
    True,
)

print(
    "Union key contract   :",
    True,
)

print(
    "Fold contract        :",
    True,
)

print(
    "R3_CELL_0_READY      :",
    R3_CELL_0_READY,
)

assert R3_CELL_0_READY is True

print("=" * 80)
print("R3 CELL 0 — FROZEN DEPENDENCY VERIFICATION: PASS")
print("=" * 80)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 0 — FROZEN R1/R2 DEPENDENCY + UNION CONTRACT VERIFICATION

ENVIRONMENT
Python  : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
OS      : Windows-10-10.0.26200-SP0
pandas  : 2.3.3
pyarrow : 25.0.1

R3 PATHS
R1 candidates : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
R1 manifest   : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_cell6_freeze_manifest.json
R2 candidates : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\frozen\r2_dense_candidates.parquet
R2 manifest   : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\frozen\r2_dense_freeze_manifest.json
R3 root       : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union

FROZEN ARTIFACT EXISTENCE
R1 candidates : True
R1 manifest   : Tr

In [4]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 1 — FROZEN R1/R2 LOAD + EXACT CANDIDATE AUDIT
# ==============================================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 1 — FROZEN R1/R2 LOAD + EXACT CANDIDATE AUDIT")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R3_CELL_0_READY is True, (
    "R3 Cell 0 must pass before Cell 1."
)

print("\nR3 Cell 0 dependency: PASS")


# ==============================================================================
# 2. EXPECTED CONTRACT
# ==============================================================================

EXPECTED_R1_ROWS = 1_752_048
EXPECTED_R2_ROWS = 1_752_048

EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821

EXPECTED_TOP_K = 50

UNION_KEYS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
]


R1_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]


R2_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 3. LOAD FROZEN R1
# ==============================================================================

print("\n" + "=" * 80)
print("LOAD FROZEN R1 CANDIDATES")
print("=" * 80)

r1_candidates = pd.read_parquet(
    R1_CANDIDATES,
    engine="pyarrow",
)

print(
    "R1 rows:",
    f"{len(r1_candidates):,}",
)

print(
    "R1 columns:",
    list(r1_candidates.columns),
)


# ==============================================================================
# 4. LOAD FROZEN R2
# ==============================================================================

print("\n" + "=" * 80)
print("LOAD FROZEN R2 CANDIDATES")
print("=" * 80)

r2_candidates = pd.read_parquet(
    R2_CANDIDATES,
    engine="pyarrow",
)

print(
    "R2 rows:",
    f"{len(r2_candidates):,}",
)

print(
    "R2 columns:",
    list(r2_candidates.columns),
)


# ==============================================================================
# 5. EXACT ROW POPULATION
# ==============================================================================

assert (
    len(r1_candidates)
    == EXPECTED_R1_ROWS
), (
    "R1 frozen candidate row count mismatch."
)

assert (
    len(r2_candidates)
    == EXPECTED_R2_ROWS
), (
    "R2 frozen candidate row count mismatch."
)


print("\n" + "=" * 80)
print("EXACT ROW POPULATION")
print("=" * 80)

print(
    "R1 observed:",
    f"{len(r1_candidates):,}",
)

print(
    "R1 expected:",
    f"{EXPECTED_R1_ROWS:,}",
)

print(
    "R2 observed:",
    f"{len(r2_candidates):,}",
)

print(
    "R2 expected:",
    f"{EXPECTED_R2_ROWS:,}",
)

print(
    "Population:",
    "PASS",
)


# ==============================================================================
# 6. SCHEMA AUDIT
# ==============================================================================

assert (
    list(r1_candidates.columns)
    == R1_REQUIRED_COLUMNS
), (
    "R1 frozen candidate schema mismatch."
)

assert (
    list(r2_candidates.columns)
    == R2_REQUIRED_COLUMNS
), (
    "R2 frozen candidate schema mismatch."
)


print("\n" + "=" * 80)
print("SCHEMA AUDIT")
print("=" * 80)

print(
    "R1 schema:",
    "PASS",
)

print(
    "R2 schema:",
    "PASS",
)


# ==============================================================================
# 7. UNION KEY NULL AUDIT
# ==============================================================================

print("\n" + "=" * 80)
print("UNION KEY NULL AUDIT")
print("=" * 80)


r1_nulls = (
    r1_candidates[
        UNION_KEYS
    ]
    .isna()
    .sum()
)

r2_nulls = (
    r2_candidates[
        UNION_KEYS
    ]
    .isna()
    .sum()
)


assert (
    int(r1_nulls.sum())
    == 0
), (
    "R1 contains null union keys."
)

assert (
    int(r2_nulls.sum())
    == 0
), (
    "R2 contains null union keys."
)


print(
    "R1 null union-key cells:",
    int(r1_nulls.sum()),
)

print(
    "R2 null union-key cells:",
    int(r2_nulls.sum()),
)

print(
    "Union-key null audit:",
    "PASS",
)


# ==============================================================================
# 8. CANDIDATE IDENTITY DUPLICATE AUDIT
# ==============================================================================

print("\n" + "=" * 80)
print("CANDIDATE IDENTITY AUDIT")
print("=" * 80)


r1_duplicate_rows = int(
    r1_candidates.duplicated(
        subset=UNION_KEYS,
        keep=False,
    ).sum()
)

r2_duplicate_rows = int(
    r2_candidates.duplicated(
        subset=UNION_KEYS,
        keep=False,
    ).sum()
)


assert (
    r1_duplicate_rows
    == 0
), (
    "R1 contains duplicate candidate identities."
)

assert (
    r2_duplicate_rows
    == 0
), (
    "R2 contains duplicate candidate identities."
)


print(
    "R1 duplicate candidate rows:",
    r1_duplicate_rows,
)

print(
    "R2 duplicate candidate rows:",
    r2_duplicate_rows,
)

print(
    "Candidate identity:",
    "PASS",
)


# ==============================================================================
# 9. RESPONSE COVERAGE
# ==============================================================================

r1_response_ids = set(
    r1_candidates[
        "response_id"
    ].astype(str)
)

r2_response_ids = set(
    r2_candidates[
        "response_id"
    ].astype(str)
)


assert (
    len(r1_response_ids)
    == EXPECTED_RESPONSES
)

assert (
    len(r2_response_ids)
    == EXPECTED_RESPONSES
)


r1_missing_from_r2 = (
    r1_response_ids
    -
    r2_response_ids
)

r2_missing_from_r1 = (
    r2_response_ids
    -
    r1_response_ids
)


assert (
    len(r1_missing_from_r2)
    == 0
), (
    "R1 contains response IDs absent from R2."
)

assert (
    len(r2_missing_from_r1)
    == 0
), (
    "R2 contains response IDs absent from R1."
)


print("\n" + "=" * 80)
print("RESPONSE COVERAGE")
print("=" * 80)

print(
    "R1 responses:",
    f"{len(r1_response_ids):,}",
)

print(
    "R2 responses:",
    f"{len(r2_response_ids):,}",
)

print(
    "R1-only responses:",
    len(r1_missing_from_r2),
)

print(
    "R2-only responses:",
    len(r2_missing_from_r1),
)

print(
    "Response coverage:",
    "PASS",
)


# ==============================================================================
# 10. SESSION COVERAGE
# ==============================================================================

r1_session_ids = set(
    r1_candidates[
        "session_id"
    ].astype(str)
)

r2_session_ids = set(
    r2_candidates[
        "session_id"
    ].astype(str)
)


assert (
    len(r1_session_ids)
    == EXPECTED_SESSIONS
)

assert (
    len(r2_session_ids)
    == EXPECTED_SESSIONS
)


assert (
    r1_session_ids
    == r2_session_ids
), (
    "R1/R2 session coverage differs."
)


print("\n" + "=" * 80)
print("SESSION COVERAGE")
print("=" * 80)

print(
    "R1 sessions:",
    f"{len(r1_session_ids):,}",
)

print(
    "R2 sessions:",
    f"{len(r2_session_ids):,}",
)

print(
    "Session identity match:",
    True,
)

print(
    "Session coverage:",
    "PASS",
)


# ==============================================================================
# 11. RESPONSE → SESSION CONSISTENCY
# ==============================================================================

r1_response_session = (
    r1_candidates[
        [
            "response_id",
            "session_id",
        ]
    ]
    .drop_duplicates()
)

r2_response_session = (
    r2_candidates[
        [
            "response_id",
            "session_id",
        ]
    ]
    .drop_duplicates()
)


r1_response_session_map = (
    r1_response_session
    .set_index(
        "response_id"
    )[
        "session_id"
    ]
    .astype(str)
)

r2_response_session_map = (
    r2_response_session
    .set_index(
        "response_id"
    )[
        "session_id"
    ]
    .astype(str)
)


assert (
    r1_response_session_map.index
    .equals(
        r2_response_session_map.index
    )
), (
    "R1/R2 response index mismatch."
)


r1_response_session_map = (
    r1_response_session_map
    .sort_index()
)

r2_response_session_map = (
    r2_response_session_map
    .sort_index()
)


assert (
    r1_response_session_map
    .equals(
        r2_response_session_map
    )
), (
    "R1/R2 response→session mapping mismatch."
)


print("\n" + "=" * 80)
print("RESPONSE → SESSION CONSISTENCY")
print("=" * 80)

print(
    "R1/R2 response→session mapping:",
    "IDENTICAL",
)

print(
    "Consistency:",
    "PASS",
)


# ==============================================================================
# 12. R1 RANK AUDIT
# ==============================================================================

assert (
    "retrieval_rank"
    not in r1_candidates.columns
), (
    "Unexpected R1 retrieval_rank column."
)

# R1's candidate order is represented by row order in the frozen artifact.
# We therefore verify population per response rather than inventing a rank.


r1_counts = (
    r1_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)


assert (
    len(r1_counts)
    == EXPECTED_RESPONSES
)

assert (
    r1_counts.min()
    >= 1
)

assert (
    r1_counts.max()
    <= EXPECTED_TOP_K
)


print("\n" + "=" * 80)
print("R1 TOP-K AUDIT")
print("=" * 80)

print(
    "Responses:",
    f"{len(r1_counts):,}",
)

print(
    "Minimum candidates/response:",
    int(r1_counts.min()),
)

print(
    "Maximum candidates/response:",
    int(r1_counts.max()),
)

print(
    "R1 Top-K population:",
    "PASS",
)


# ==============================================================================
# 13. R2 RANK AUDIT
# ==============================================================================

r2_rank = (
    r2_candidates[
        "retrieval_rank"
    ]
    .astype(int)
)


assert (
    r2_rank.min()
    >= 0
)

assert (
    r2_rank.max()
    < EXPECTED_TOP_K
)


r2_rank_counts = (
    r2_candidates
    .groupby(
        "response_id",
        sort=False,
    )[
        "retrieval_rank"
    ]
    .nunique()
)


assert (
    r2_rank_counts.max()
    <= EXPECTED_TOP_K
)


print("\n" + "=" * 80)
print("R2 TOP-K AUDIT")
print("=" * 80)

print(
    "Rank minimum:",
    int(r2_rank.min()),
)

print(
    "Rank maximum:",
    int(r2_rank.max()),
)

print(
    "Maximum unique ranks/response:",
    int(r2_rank_counts.max()),
)

print(
    "R2 Top-K population:",
    "PASS",
)


# ==============================================================================
# 14. SCORE FINITENESS
# ==============================================================================

r1_score_columns = [
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

r2_score_columns = [
    "dense_score",
]


for column in r1_score_columns:

    values = (
        r1_candidates[
            column
        ]
        .astype(float)
        .to_numpy()
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite R1 score: {column}"
    )


for column in r2_score_columns:

    values = (
        r2_candidates[
            column
        ]
        .astype(float)
        .to_numpy()
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite R2 score: {column}"
    )


print("\n" + "=" * 80)
print("SCORE VALIDITY")
print("=" * 80)

print(
    "R1 score columns:",
    r1_score_columns,
)

print(
    "R2 score columns:",
    r2_score_columns,
)

print(
    "Finite values:",
    "PASS",
)


# ==============================================================================
# 15. CROSS-SESSION CONTAMINATION
# ==============================================================================

# A response must map to exactly one session within each frozen source.
# This is an explicit R3 safety gate.


r1_multi_session = int(
    (
        r1_candidates
        .groupby(
            "response_id"
        )[
            "session_id"
        ]
        .nunique()
        > 1
    ).sum()
)


r2_multi_session = int(
    (
        r2_candidates
        .groupby(
            "response_id"
        )[
            "session_id"
        ]
        .nunique()
        > 1
    ).sum()
)


assert (
    r1_multi_session
    == 0
)

assert (
    r2_multi_session
    == 0
)


print("\n" + "=" * 80)
print("SESSION ISOLATION")
print("=" * 80)

print(
    "R1 multi-session responses:",
    r1_multi_session,
)

print(
    "R2 multi-session responses:",
    r2_multi_session,
)

print(
    "Cross-session contamination:",
    "PASS",
)


# ==============================================================================
# 16. FOLD CONSISTENCY
# ==============================================================================

r1_response_fold = (
    r1_candidates[
        [
            "response_id",
            "fold",
        ]
    ]
    .drop_duplicates()
)

r2_response_fold = (
    r2_candidates[
        [
            "response_id",
            "fold",
        ]
    ]
    .drop_duplicates()
)


r1_response_fold_map = (
    r1_response_fold
    .groupby(
        "response_id"
    )[
        "fold"
    ]
    .nunique()
)

r2_response_fold_map = (
    r2_response_fold
    .groupby(
        "response_id"
    )[
        "fold"
    ]
    .nunique()
)


assert (
    r1_response_fold_map.max()
    == 1
)

assert (
    r2_response_fold_map.max()
    == 1
)


r1_folds = sorted(
    r1_candidates[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)

r2_folds = sorted(
    r2_candidates[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert (
    r1_folds
    == EXPECTED_FOLDS
)

assert (
    r2_folds
    == EXPECTED_FOLDS
)

assert (
    r1_folds
    == r2_folds
)


print("\n" + "=" * 80)
print("FOLD CONSISTENCY")
print("=" * 80)

print(
    "R1 folds:",
    r1_folds,
)

print(
    "R2 folds:",
    r2_folds,
)

print(
    "Fold consistency:",
    "PASS",
)


# ==============================================================================
# 17. TARGET ISOLATION
# ==============================================================================

assert not (
    PROHIBITED_COLUMNS
    &
    set(r1_candidates.columns)
)

assert not (
    PROHIBITED_COLUMNS
    &
    set(r2_candidates.columns)
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "R1 target columns:",
    "NONE",
)

print(
    "R2 target columns:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 18. R3 CELL 1 FINAL GATE
# ==============================================================================

R3_CELL_1_READY = True


print("\n" + "=" * 80)
print("R3 CELL 1 STATUS")
print("=" * 80)

print(
    "R1 population:",
    f"{len(r1_candidates):,}",
)

print(
    "R2 population:",
    f"{len(r2_candidates):,}",
)

print(
    "Responses:",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "Sessions:",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "Response coverage:",
    "PASS",
)

print(
    "Session coverage:",
    "PASS",
)

print(
    "Candidate identity:",
    "PASS",
)

print(
    "Score validity:",
    "PASS",
)

print(
    "Fold consistency:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "R3_CELL_1_READY:",
    R3_CELL_1_READY,
)

assert R3_CELL_1_READY is True

print("=" * 80)
print("R3 CELL 1 — FROZEN CANDIDATE AUDIT: PASS")
print("=" * 80)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 1 — FROZEN R1/R2 LOAD + EXACT CANDIDATE AUDIT

R3 Cell 0 dependency: PASS

LOAD FROZEN R1 CANDIDATES
R1 rows: 1,752,048
R1 columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'word_score', 'char_score', 'math_score', 'sparse_score']

LOAD FROZEN R2 CANDIDATES
R2 rows: 1,752,048
R2 columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'dense_score', 'retrieval_rank', 'text_norm']

EXACT ROW POPULATION
R1 observed: 1,752,048
R1 expected: 1,752,048
R2 observed: 1,752,048
R2 expected: 1,752,048
Population: PASS

SCHEMA AUDIT
R1 schema: PASS
R2 schema: PASS

UNION KEY NULL AUDIT
R1 null union-key cells: 0
R2 null union-key cells: 0
Union-key null audit: PASS

CANDIDATE IDENTITY AUDIT
R1 duplicate candidate rows: 0
R2 duplicate candidate rows: 0
Candidate identity: PASS

RESPONSE COVERAGE
R1 responses: 35,072
R2 responses: 35,072
R1-only responses: 0
R2-only respo

In [6]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 2 — SPARSE + DENSE CANDIDATE UNION
# ==============================================================================

import gc
import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 2 — SPARSE + DENSE CANDIDATE UNION")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R3_CELL_0_READY is True, (
    "R3 Cell 0 must pass before Cell 2."
)

assert R3_CELL_1_READY is True, (
    "R3 Cell 1 must pass before Cell 2."
)

print("\nR3 Cell 0 dependency: PASS")
print("R3 Cell 1 dependency: PASS")


# ==============================================================================
# 2. SOURCE POPULATION
# ==============================================================================

assert len(r1_candidates) == 1_752_048
assert len(r2_candidates) == 1_752_048


print("\n" + "=" * 80)
print("SOURCE POPULATION")
print("=" * 80)

print(
    "R1 sparse candidates:",
    f"{len(r1_candidates):,}",
)

print(
    "R2 dense candidates :",
    f"{len(r2_candidates):,}",
)


# ==============================================================================
# 3. DEFINE CANONICAL CANDIDATE IDENTITY
# ==============================================================================

CANDIDATE_ID_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
]


# ==============================================================================
# 4. PREPARE R1
# ==============================================================================

print("\n" + "=" * 80)
print("PREPARE R1 SPARSE CANDIDATES")
print("=" * 80)


r1_union = r1_candidates[
    [
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "turn_uid",
        "role",
        "turn_index",
        "word_score",
        "char_score",
        "math_score",
        "sparse_score",
    ]
].copy()


# Preserve the exact frozen R1 row order as deterministic tie-breaker.
r1_union["_r1_source_order"] = np.arange(
    len(r1_union),
    dtype=np.int64,
)


# ------------------------------------------------------------------------------
# Derive explicit sparse_rank
#
# R1 did not persist a rank column.
#
# Ranking rule:
#   1. response-local sparse_score descending
#   2. original frozen R1 row order as deterministic tie-break
#
# Rank is zero-based to match R2 retrieval_rank semantics.
# ------------------------------------------------------------------------------

r1_union = r1_union.sort_values(
    [
        "response_id",
        "sparse_score",
        "_r1_source_order",
    ],
    ascending=[
        True,
        False,
        True,
    ],
    kind="mergesort",
)


r1_union[
    "sparse_rank"
] = (
    r1_union
    .groupby(
        "response_id",
        sort=False,
    )
    .cumcount()
    .astype(np.int16)
)


assert (
    r1_union[
        "sparse_rank"
    ].min()
    == 0
)

assert (
    r1_union[
        "sparse_rank"
    ].max()
    < 50
)


# Sparse provenance flag.
r1_union[
    "selected_sparse"
] = True


# R1 does not carry dense fields.
r1_union[
    "dense_score"
] = np.nan

r1_union[
    "dense_rank"
] = np.nan

r1_union[
    "selected_dense"
] = False


# R1 does not carry text_norm.
r1_union[
    "text_norm"
] = pd.NA


# Source marker.
r1_union[
    "_from_sparse"
] = True

r1_union[
    "_from_dense"
] = False


print(
    "R1 sparse rank derived:",
    "PASS",
)

print(
    "R1 sparse rank range:",
    int(r1_union["sparse_rank"].min()),
    "→",
    int(r1_union["sparse_rank"].max()),
)


# ==============================================================================
# 5. PREPARE R2
# ==============================================================================

print("\n" + "=" * 80)
print("PREPARE R2 DENSE CANDIDATES")
print("=" * 80)


r2_union = r2_candidates[
    [
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "turn_uid",
        "role",
        "turn_index",
        "dense_score",
        "retrieval_rank",
        "text_norm",
    ]
].copy()


# Normalize R2 rank naming for R3.
r2_union[
    "dense_rank"
] = (
    r2_union[
        "retrieval_rank"
    ]
    .astype(np.int16)
)


r2_union = r2_union.drop(
    columns=[
        "retrieval_rank",
    ]
)


# R2 does not carry sparse fields.
r2_union[
    "word_score"
] = np.nan

r2_union[
    "char_score"
] = np.nan

r2_union[
    "math_score"
] = np.nan

r2_union[
    "sparse_score"
] = np.nan

r2_union[
    "sparse_rank"
] = np.nan


# Dense provenance flag.
r2_union[
    "selected_dense"
] = True


r2_union[
    "selected_sparse"
] = False


# Source markers.
r2_union[
    "_from_sparse"
] = False

r2_union[
    "_from_dense"
] = True


assert (
    r2_union[
        "dense_rank"
    ].min()
    == 0
)

assert (
    r2_union[
        "dense_rank"
    ].max()
    < 50
)


print(
    "R2 dense rank loaded:",
    "PASS",
)

print(
    "R2 dense rank range:",
    int(r2_union["dense_rank"].min()),
    "→",
    int(r2_union["dense_rank"].max()),
)


# ==============================================================================
# 6. ALIGN COLUMN ORDER
# ==============================================================================

R3_UNION_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "text_norm",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
    "sparse_rank",
    "dense_score",
    "dense_rank",
    "selected_sparse",
    "selected_dense",
    "_from_sparse",
    "_from_dense",
]


r1_union = r1_union[
    R3_UNION_COLUMNS
]

r2_union = r2_union[
    R3_UNION_COLUMNS
]


# ==============================================================================
# 7. CONCATENATE — NO DEDUP YET
# ==============================================================================

print("\n" + "=" * 80)
print("RAW SPARSE + DENSE UNION INPUT")
print("=" * 80)


raw_union_rows = (
    len(r1_union)
    +
    len(r2_union)
)


assert (
    raw_union_rows
    == 3_504_096
)


r3_raw_union = pd.concat(
    [
        r1_union,
        r2_union,
    ],
    axis=0,
    ignore_index=True,
)


assert (
    len(r3_raw_union)
    == raw_union_rows
)


print(
    "R1 rows:",
    f"{len(r1_union):,}",
)

print(
    "R2 rows:",
    f"{len(r2_union):,}",
)

print(
    "Raw combined rows:",
    f"{len(r3_raw_union):,}",
)


# ==============================================================================
# 8. PRE-DEDUP OVERLAP AUDIT
# ==============================================================================

print("\n" + "=" * 80)
print("SPARSE / DENSE OVERLAP")
print("=" * 80)


r1_keys = pd.MultiIndex.from_frame(
    r1_union[
        CANDIDATE_ID_COLUMNS
    ]
)

r2_keys = pd.MultiIndex.from_frame(
    r2_union[
        CANDIDATE_ID_COLUMNS
    ]
)


# Both frozen sources were already verified to have unique candidate identities
# in Cell 1. Therefore set-style intersection is valid here.

overlap_keys = r1_keys.intersection(
    r2_keys
)

overlap_count = len(
    overlap_keys
)


r1_only_count = (
    len(r1_keys)
    -
    overlap_count
)

r2_only_count = (
    len(r2_keys)
    -
    overlap_count
)


# --------------------------------------------------------------------------
# Correct accounting identities
#
# RAW UNION:
#     R1 + R2
#
# UNIQUE UNION:
#     R1-only + R2-only + overlap
#
# Therefore:
#     unique_union = raw_union - overlap
# --------------------------------------------------------------------------

expected_unique_rows = (
    r1_only_count
    +
    r2_only_count
    +
    overlap_count
)

expected_unique_rows_from_raw = (
    len(r3_raw_union)
    -
    overlap_count
)


assert (
    expected_unique_rows
    ==
    expected_unique_rows_from_raw
), (
    "Sparse/dense overlap accounting mismatch."
)


assert (
    r1_only_count
    +
    overlap_count
    ==
    len(r1_keys)
), (
    "R1 overlap accounting mismatch."
)


assert (
    r2_only_count
    +
    overlap_count
    ==
    len(r2_keys)
), (
    "R2 overlap accounting mismatch."
)


print(
    "R1 candidates:",
    f"{len(r1_keys):,}",
)

print(
    "R2 candidates:",
    f"{len(r2_keys):,}",
)

print(
    "Raw combined rows:",
    f"{len(r3_raw_union):,}",
)

print(
    "Overlap:",
    f"{overlap_count:,}",
)

print(
    "R1-only:",
    f"{r1_only_count:,}",
)

print(
    "R2-only:",
    f"{r2_only_count:,}",
)

print(
    "Expected unique union:",
    f"{expected_unique_rows:,}",
)

print(
    "Overlap accounting:",
    "PASS",
)


# ==============================================================================
# 9. DEDUPLICATE BY CANONICAL CANDIDATE IDENTITY
# ==============================================================================

print("\n" + "=" * 80)
print("CANDIDATE DEDUPLICATION")
print("=" * 80)


# ------------------------------------------------------------------------------
# The canonical candidate identity is:
#
# response_id
# session_id
# objective_uid
# fold
# turn_uid
#
# One physical turn may be retrieved by both sparse and dense retrieval.
# Such a turn must become ONE R3 candidate row.
# ------------------------------------------------------------------------------

r3_union = (
    r3_raw_union
    .groupby(
        CANDIDATE_ID_COLUMNS,
        sort=False,
        dropna=False,
        as_index=False,
    )
    .agg(
        {
            "role": "first",
            "turn_index": "first",
            "text_norm": "first",

            "word_score": "max",
            "char_score": "max",
            "math_score": "max",
            "sparse_score": "max",
            "sparse_rank": "min",

            "dense_score": "max",
            "dense_rank": "min",

            "selected_sparse": "max",
            "selected_dense": "max",

            "_from_sparse": "max",
            "_from_dense": "max",
        }
    )
)


# ==============================================================================
# 10. NORMALIZE PROVENANCE FLAGS
# ==============================================================================

r3_union[
    "selected_sparse"
] = (
    r3_union[
        "selected_sparse"
    ]
    .astype(bool)
)

r3_union[
    "selected_dense"
] = (
    r3_union[
        "selected_dense"
    ]
    .astype(bool)
)

r3_union[
    "_from_sparse"
] = (
    r3_union[
        "_from_sparse"
    ]
    .astype(bool)
)

r3_union[
    "_from_dense"
] = (
    r3_union[
        "_from_dense"
    ]
    .astype(bool)
)


# Candidate union is true for every row by construction.
r3_union[
    "candidate_union"
] = True


# ==============================================================================
# 11. VERIFY DEDUPLICATION
# ==============================================================================

post_dedup_duplicates = int(
    r3_union.duplicated(
        subset=CANDIDATE_ID_COLUMNS,
        keep=False,
    ).sum()
)


assert (
    post_dedup_duplicates
    == 0
)


expected_unique_rows = (
    r1_only_count
    +
    r2_only_count
    +
    overlap_count
)


assert (
    len(r3_union)
    == expected_unique_rows
)


assert (
    len(r3_union)
    ==
    len(r3_raw_union)
    -
    overlap_count
)


print(
    "Raw rows:",
    f"{len(r3_raw_union):,}",
)

print(
    "Overlap rows removed:",
    f"{overlap_count:,}",
)

print(
    "Unique union rows:",
    f"{len(r3_union):,}",
)

print(
    "Post-dedup duplicate rows:",
    post_dedup_duplicates,
)


# ==============================================================================
# 12. PROVENANCE AUDIT
# ==============================================================================

print("\n" + "=" * 80)
print("CANDIDATE PROVENANCE")
print("=" * 80)


sparse_only = int(
    (
        r3_union[
            "selected_sparse"
        ]
        &
        ~r3_union[
            "selected_dense"
        ]
    ).sum()
)


dense_only = int(
    (
        ~r3_union[
            "selected_sparse"
        ]
        &
        r3_union[
            "selected_dense"
        ]
    ).sum()
)


both_sources = int(
    (
        r3_union[
            "selected_sparse"
        ]
        &
        r3_union[
            "selected_dense"
        ]
    ).sum()
)


provenance_total = (
    sparse_only
    +
    dense_only
    +
    both_sources
)


assert (
    provenance_total
    == len(r3_union)
)


assert (
    both_sources
    == overlap_count
)


print(
    "Sparse-only candidates:",
    f"{sparse_only:,}",
)

print(
    "Dense-only candidates:",
    f"{dense_only:,}",
)

print(
    "Both sparse + dense:",
    f"{both_sources:,}",
)

print(
    "Total union candidates:",
    f"{provenance_total:,}",
)


# ==============================================================================
# 13. SAME-SESSION SAFETY AFTER UNION
# ==============================================================================

response_session_counts = (
    r3_union
    .groupby(
        "response_id",
        sort=False,
    )[
        "session_id"
    ]
    .nunique()
)


session_contamination = int(
    (
        response_session_counts
        > 1
    ).sum()
)


assert (
    session_contamination
    == 0
)


print("\n" + "=" * 80)
print("POST-UNION SESSION SAFETY")
print("=" * 80)

print(
    "Responses with multiple sessions:",
    session_contamination,
)

print(
    "Cross-session contamination:",
    "PASS",
)


# ==============================================================================
# 14. RESPONSE COVERAGE AFTER UNION
# ==============================================================================

r3_response_ids = set(
    r3_union[
        "response_id"
    ].astype(str)
)


assert (
    len(r3_response_ids)
    == EXPECTED_RESPONSES
)

assert (
    r3_response_ids
    ==
    r1_response_ids
)

assert (
    r3_response_ids
    ==
    r2_response_ids
)


print("\n" + "=" * 80)
print("POST-UNION RESPONSE COVERAGE")
print("=" * 80)

print(
    "R3 responses:",
    f"{len(r3_response_ids):,}",
)

print(
    "Expected:",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "Response coverage:",
    "PASS",
)


# ==============================================================================
# 15. CANDIDATE UNION FLAG
# ==============================================================================

assert (
    r3_union[
        "candidate_union"
    ].all()
)


print(
    "candidate_union:",
    "True for all rows",
)


# ==============================================================================
# 16. R3 CELL 2 STATUS
# ==============================================================================

R3_CELL_2_READY = True


print("\n" + "=" * 80)
print("R3 CELL 2 STATUS")
print("=" * 80)

print(
    "R1 source rows:",
    f"{len(r1_union):,}",
)

print(
    "R2 source rows:",
    f"{len(r2_union):,}",
)

print(
    "Raw combined rows:",
    f"{len(r3_raw_union):,}",
)

print(
    "Sparse/Dense overlap:",
    f"{overlap_count:,}",
)

print(
    "Unique union candidates:",
    f"{len(r3_union):,}",
)

print(
    "Sparse-only:",
    f"{sparse_only:,}",
)

print(
    "Dense-only:",
    f"{dense_only:,}",
)

print(
    "Both sources:",
    f"{both_sources:,}",
)

print(
    "Cross-session contamination:",
    session_contamination,
)

print(
    "Response coverage:",
    "PASS",
)

print(
    "R3_CELL_2_READY:",
    R3_CELL_2_READY,
)

assert R3_CELL_2_READY is True

print("=" * 80)
print("R3 CELL 2 — CANDIDATE UNION: PASS")
print("=" * 80)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 2 — SPARSE + DENSE CANDIDATE UNION

R3 Cell 0 dependency: PASS
R3 Cell 1 dependency: PASS

SOURCE POPULATION
R1 sparse candidates: 1,752,048
R2 dense candidates : 1,752,048

PREPARE R1 SPARSE CANDIDATES
R1 sparse rank derived: PASS
R1 sparse rank range: 0 → 49

PREPARE R2 DENSE CANDIDATES
R2 dense rank loaded: PASS
R2 dense rank range: 0 → 49

RAW SPARSE + DENSE UNION INPUT


C:\Users\USER\AppData\Local\Temp\ipykernel_26748\1020381067.py:378: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  r3_raw_union = pd.concat(


R1 rows: 1,752,048
R2 rows: 1,752,048
Raw combined rows: 3,504,096

SPARSE / DENSE OVERLAP
R1 candidates: 1,752,048
R2 candidates: 1,752,048
Raw combined rows: 3,504,096
Overlap: 1,021,959
R1-only: 730,089
R2-only: 730,089
Expected unique union: 2,482,137
Overlap accounting: PASS

CANDIDATE DEDUPLICATION
Raw rows: 3,504,096
Overlap rows removed: 1,021,959
Unique union rows: 2,482,137
Post-dedup duplicate rows: 0

CANDIDATE PROVENANCE
Sparse-only candidates: 730,089
Dense-only candidates: 730,089
Both sparse + dense: 1,021,959
Total union candidates: 2,482,137

POST-UNION SESSION SAFETY
Responses with multiple sessions: 0
Cross-session contamination: PASS

POST-UNION RESPONSE COVERAGE
R3 responses: 35,072
Expected: 35,072
Response coverage: PASS
candidate_union: True for all rows

R3 CELL 2 STATUS
R1 source rows: 1,752,048
R2 source rows: 1,752,048
Raw combined rows: 3,504,096
Sparse/Dense overlap: 1,021,959
Unique union candidates: 2,482,137
Sparse-only: 730,089
Dense-only: 730,089
Bot

In [7]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 3 — UNION INTEGRITY + PROVENANCE AUDIT
# ==============================================================================

import gc
import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 3 — UNION INTEGRITY + PROVENANCE AUDIT")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R3_CELL_0_READY is True, (
    "R3 Cell 0 must pass."
)

assert R3_CELL_1_READY is True, (
    "R3 Cell 1 must pass."
)

assert R3_CELL_2_READY is True, (
    "R3 Cell 2 must pass."
)


print("\nR3 Cell 0 dependency: PASS")
print("R3 Cell 1 dependency: PASS")
print("R3 Cell 2 dependency: PASS")


# ==============================================================================
# 2. EXPECTED UNION POPULATION
# ==============================================================================

EXPECTED_R1_ROWS = 1_752_048
EXPECTED_R2_ROWS = 1_752_048
EXPECTED_OVERLAP = 1_021_959
EXPECTED_UNION_ROWS = 2_482_137
EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821


assert len(r3_union) == EXPECTED_UNION_ROWS


print("\n" + "=" * 80)
print("UNION POPULATION")
print("=" * 80)

print(
    "R1 source rows:",
    f"{len(r1_candidates):,}",
)

print(
    "R2 source rows:",
    f"{len(r2_candidates):,}",
)

print(
    "Expected overlap:",
    f"{EXPECTED_OVERLAP:,}",
)

print(
    "Observed union rows:",
    f"{len(r3_union):,}",
)

print(
    "Expected union rows:",
    f"{EXPECTED_UNION_ROWS:,}",
)


# ==============================================================================
# 3. CANONICAL IDENTITY UNIQUENESS
# ==============================================================================

CANDIDATE_ID_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
]


duplicate_identity_rows = int(
    r3_union.duplicated(
        subset=CANDIDATE_ID_COLUMNS,
        keep=False,
    ).sum()
)


assert duplicate_identity_rows == 0, (
    "R3 union contains duplicate candidate identities."
)


print("\n" + "=" * 80)
print("CANONICAL CANDIDATE IDENTITY")
print("=" * 80)

print(
    "Identity columns:",
    CANDIDATE_ID_COLUMNS,
)

print(
    "Duplicate identity rows:",
    duplicate_identity_rows,
)

print(
    "Candidate identity:",
    "PASS",
)


# ==============================================================================
# 4. REQUIRED R3 SCHEMA
# ==============================================================================

R3_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "text_norm",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
    "sparse_rank",
    "dense_score",
    "dense_rank",
    "selected_sparse",
    "selected_dense",
    "_from_sparse",
    "_from_dense",
    "candidate_union",
]


missing_r3_columns = [
    column
    for column in R3_REQUIRED_COLUMNS
    if column not in r3_union.columns
]


assert not missing_r3_columns, (
    f"Missing R3 columns: {missing_r3_columns}"
)


print("\n" + "=" * 80)
print("R3 SCHEMA")
print("=" * 80)

print(
    "Required columns:",
    len(R3_REQUIRED_COLUMNS),
)

print(
    "Missing columns:",
    missing_r3_columns,
)

print(
    "Schema contract:",
    "PASS",
)


# ==============================================================================
# 5. RESPONSE COVERAGE
# ==============================================================================

r3_response_ids = set(
    r3_union[
        "response_id"
    ].astype(str)
)

assert (
    len(r3_response_ids)
    == EXPECTED_RESPONSES
)


assert (
    r3_response_ids
    == r1_response_ids
)


assert (
    r3_response_ids
    == r2_response_ids
)


print("\n" + "=" * 80)
print("RESPONSE COVERAGE")
print("=" * 80)

print(
    "R3 responses:",
    f"{len(r3_response_ids):,}",
)

print(
    "Expected responses:",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "R1/R3 response identity:",
    "MATCH",
)

print(
    "R2/R3 response identity:",
    "MATCH",
)

print(
    "Response coverage:",
    "PASS",
)


# ==============================================================================
# 6. SESSION COVERAGE
# ==============================================================================

r3_session_ids = set(
    r3_union[
        "session_id"
    ].astype(str)
)


assert (
    len(r3_session_ids)
    == EXPECTED_SESSIONS
)

assert (
    r3_session_ids
    == r1_session_ids
)

assert (
    r3_session_ids
    == r2_session_ids
)


print("\n" + "=" * 80)
print("SESSION COVERAGE")
print("=" * 80)

print(
    "R3 sessions:",
    f"{len(r3_session_ids):,}",
)

print(
    "Expected sessions:",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "Session coverage:",
    "PASS",
)


# ==============================================================================
# 7. RESPONSE → SESSION CONSISTENCY
# ==============================================================================

r3_response_session_counts = (
    r3_union
    .groupby(
        "response_id",
        sort=False,
    )[
        "session_id"
    ]
    .nunique()
)


multi_session_responses = int(
    (
        r3_response_session_counts
        > 1
    ).sum()
)


assert (
    multi_session_responses
    == 0
)


print("\n" + "=" * 80)
print("RESPONSE → SESSION CONSISTENCY")
print("=" * 80)

print(
    "Responses mapped to >1 session:",
    multi_session_responses,
)

print(
    "Cross-session contamination:",
    "PASS",
)


# ==============================================================================
# 8. PROVENANCE FLAGS
# ==============================================================================

assert (
    r3_union[
        "selected_sparse"
    ].dtype
    == bool
)

assert (
    r3_union[
        "selected_dense"
    ].dtype
    == bool
)


sparse_only = int(
    (
        r3_union["selected_sparse"]
        &
        ~r3_union["selected_dense"]
    ).sum()
)


dense_only = int(
    (
        ~r3_union["selected_sparse"]
        &
        r3_union["selected_dense"]
    ).sum()
)


both_sources = int(
    (
        r3_union["selected_sparse"]
        &
        r3_union["selected_dense"]
    ).sum()
)


assert (
    sparse_only
    +
    dense_only
    +
    both_sources
    ==
    len(r3_union)
)


assert (
    both_sources
    == EXPECTED_OVERLAP
)


print("\n" + "=" * 80)
print("PROVENANCE")
print("=" * 80)

print(
    "Sparse-only:",
    f"{sparse_only:,}",
)

print(
    "Dense-only:",
    f"{dense_only:,}",
)

print(
    "Both:",
    f"{both_sources:,}",
)

print(
    "Union total:",
    f"{len(r3_union):,}",
)

print(
    "Provenance accounting:",
    "PASS",
)


# ==============================================================================
# 9. SOURCE MARKER CONSISTENCY
# ==============================================================================

assert (
    r3_union[
        "_from_sparse"
    ]
    ==
    r3_union[
        "selected_sparse"
    ]
).all()


assert (
    r3_union[
        "_from_dense"
    ]
    ==
    r3_union[
        "selected_dense"
    ]
).all()


print("\n" + "=" * 80)
print("SOURCE MARKER CONSISTENCY")
print("=" * 80)

print(
    "_from_sparse ↔ selected_sparse:",
    "MATCH",
)

print(
    "_from_dense ↔ selected_dense:",
    "MATCH",
)

print(
    "Source marker contract:",
    "PASS",
)


# ==============================================================================
# 10. SPARSE-ONLY SCORE/RANK INTEGRITY
# ==============================================================================

sparse_only_mask = (
    r3_union[
        "selected_sparse"
    ]
    &
    ~r3_union[
        "selected_dense"
    ]
)


sparse_only_rows = r3_union[
    sparse_only_mask
]


assert (
    sparse_only_rows[
        "sparse_score"
    ].notna().all()
)


assert (
    sparse_only_rows[
        "sparse_rank"
    ].notna().all()
)


assert (
    sparse_only_rows[
        "dense_score"
    ].isna().all()
)


assert (
    sparse_only_rows[
        "dense_rank"
    ].isna().all()
)


print("\n" + "=" * 80)
print("SPARSE-ONLY INTEGRITY")
print("=" * 80)

print(
    "Sparse-only rows:",
    f"{len(sparse_only_rows):,}",
)

print(
    "Sparse score present:",
    True,
)

print(
    "Sparse rank present:",
    True,
)

print(
    "Dense score absent:",
    True,
)

print(
    "Dense rank absent:",
    True,
)

print(
    "Sparse-only integrity:",
    "PASS",
)


# ==============================================================================
# 11. DENSE-ONLY SCORE/RANK INTEGRITY
# ==============================================================================

dense_only_mask = (
    ~r3_union[
        "selected_sparse"
    ]
    &
    r3_union[
        "selected_dense"
    ]
)


dense_only_rows = r3_union[
    dense_only_mask
]


assert (
    dense_only_rows[
        "dense_score"
    ].notna().all()
)


assert (
    dense_only_rows[
        "dense_rank"
    ].notna().all()
)


assert (
    dense_only_rows[
        "sparse_score"
    ].isna().all()
)


assert (
    dense_only_rows[
        "sparse_rank"
    ].isna().all()
)


print("\n" + "=" * 80)
print("DENSE-ONLY INTEGRITY")
print("=" * 80)

print(
    "Dense-only rows:",
    f"{len(dense_only_rows):,}",
)

print(
    "Dense score present:",
    True,
)

print(
    "Dense rank present:",
    True,
)

print(
    "Sparse score absent:",
    True,
)

print(
    "Sparse rank absent:",
    True,
)

print(
    "Dense-only integrity:",
    "PASS",
)


# ==============================================================================
# 12. BOTH-SOURCE SCORE/RANK INTEGRITY
# ==============================================================================

both_mask = (
    r3_union[
        "selected_sparse"
    ]
    &
    r3_union[
        "selected_dense"
    ]
)


both_rows = r3_union[
    both_mask
]


assert (
    both_rows[
        "sparse_score"
    ].notna().all()
)


assert (
    both_rows[
        "sparse_rank"
    ].notna().all()
)


assert (
    both_rows[
        "dense_score"
    ].notna().all()
)


assert (
    both_rows[
        "dense_rank"
    ].notna().all()
)


print("\n" + "=" * 80)
print("BOTH-SOURCE INTEGRITY")
print("=" * 80)

print(
    "Both-source rows:",
    f"{len(both_rows):,}",
)

print(
    "Sparse score present:",
    True,
)

print(
    "Sparse rank present:",
    True,
)

print(
    "Dense score present:",
    True,
)

print(
    "Dense rank present:",
    True,
)

print(
    "Both-source integrity:",
    "PASS",
)


# ==============================================================================
# 13. SCORE FINITENESS
# ==============================================================================

R3_SCORE_COLUMNS = [
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
    "dense_score",
]


for column in R3_SCORE_COLUMNS:

    values = (
        r3_union[
            column
        ]
        .dropna()
        .astype(float)
        .to_numpy()
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite values in {column}."
    )


print("\n" + "=" * 80)
print("SCORE FINITENESS")
print("=" * 80)

print(
    "Checked:",
    R3_SCORE_COLUMNS,
)

print(
    "Finite non-null scores:",
    "PASS",
)


# ==============================================================================
# 14. RANK RANGE
# ==============================================================================

sparse_rank_values = (
    r3_union[
        "sparse_rank"
    ]
    .dropna()
    .astype(int)
)

dense_rank_values = (
    r3_union[
        "dense_rank"
    ]
    .dropna()
    .astype(int)
)


assert (
    sparse_rank_values.min()
    >= 0
)

assert (
    sparse_rank_values.max()
    < 50
)

assert (
    dense_rank_values.min()
    >= 0
)

assert (
    dense_rank_values.max()
    < 50
)


print("\n" + "=" * 80)
print("RANK CONTRACT")
print("=" * 80)

print(
    "Sparse rank range:",
    int(sparse_rank_values.min()),
    "→",
    int(sparse_rank_values.max()),
)

print(
    "Dense rank range:",
    int(dense_rank_values.min()),
    "→",
    int(dense_rank_values.max()),
)

print(
    "Rank contract:",
    "PASS",
)


# ==============================================================================
# 15. TURN METADATA CONSISTENCY
# ==============================================================================

# R1 and R2 should describe the same physical turn identically.
#
# Compare only overlapping candidate identities.
# This catches accidental identity collisions or metadata drift during union.

r1_overlap = r1_union[
    r1_union[
        CANDIDATE_ID_COLUMNS
    ].apply(
        tuple,
        axis=1,
    ).isin(
        set(overlap_keys)
    )
].copy()


r2_overlap = r2_union[
    r2_union[
        CANDIDATE_ID_COLUMNS
    ].apply(
        tuple,
        axis=1,
    ).isin(
        set(overlap_keys)
    )
].copy()


r1_overlap_check = r1_overlap[
    CANDIDATE_ID_COLUMNS
    +
    [
        "role",
        "turn_index",
    ]
].copy()


r2_overlap_check = r2_overlap[
    CANDIDATE_ID_COLUMNS
    +
    [
        "role",
        "turn_index",
    ]
].copy()


r1_overlap_check = (
    r1_overlap_check
    .sort_values(
        CANDIDATE_ID_COLUMNS
    )
    .reset_index(
        drop=True
    )
)


r2_overlap_check = (
    r2_overlap_check
    .sort_values(
        CANDIDATE_ID_COLUMNS
    )
    .reset_index(
        drop=True
    )
)


assert (
    r1_overlap_check[
        CANDIDATE_ID_COLUMNS
        +
        [
            "role",
            "turn_index",
        ]
    ]
    .equals(
        r2_overlap_check[
            CANDIDATE_ID_COLUMNS
            +
            [
                "role",
                "turn_index",
            ]
        ]
    )
)


print("\n" + "=" * 80)
print("OVERLAPPING TURN METADATA")
print("=" * 80)

print(
    "Overlap candidates:",
    f"{len(r1_overlap_check):,}",
)

print(
    "Role consistency:",
    "PASS",
)

print(
    "Turn-index consistency:",
    "PASS",
)

print(
    "Physical-turn identity:",
    "PASS",
)


# ==============================================================================
# 16. OBJECTIVE / FOLD CONSISTENCY
# ==============================================================================

r3_objectives = (
    r3_union[
        "objective_uid"
    ]
    .nunique()
)

r3_folds = sorted(
    r3_union[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert (
    r3_objectives
    == 398
)


assert (
    r3_folds
    == [0, 1, 2, 3, 4]
)


print("\n" + "=" * 80)
print("OBJECTIVE / FOLD CONTRACT")
print("=" * 80)

print(
    "Objectives:",
    r3_objectives,
)

print(
    "Folds:",
    r3_folds,
)

print(
    "Objective/fold contract:",
    "PASS",
)


# ==============================================================================
# 17. TARGET ISOLATION
# ==============================================================================

PROHIBITED_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
    "prediction",
}


target_columns_present = (
    PROHIBITED_COLUMNS
    &
    set(r3_union.columns)
)


assert not target_columns_present, (
    "R3 contains prohibited target/prediction columns."
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Prohibited columns found:",
    sorted(target_columns_present),
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 18. CANDIDATE UNION FLAG
# ==============================================================================

assert (
    r3_union[
        "candidate_union"
    ]
    .astype(bool)
    .all()
)


print(
    "candidate_union flag:",
    "PASS",
)


# ==============================================================================
# 19. FINAL R3 CELL 3 GATE
# ==============================================================================

R3_CELL_3_READY = True


print("\n" + "=" * 80)
print("R3 CELL 3 STATUS")
print("=" * 80)

print(
    "Union rows:",
    f"{len(r3_union):,}",
)

print(
    "Sparse-only:",
    f"{sparse_only:,}",
)

print(
    "Dense-only:",
    f"{dense_only:,}",
)

print(
    "Both-source:",
    f"{both_sources:,}",
)

print(
    "Duplicate identities:",
    duplicate_identity_rows,
)

print(
    "Cross-session contamination:",
    multi_session_responses,
)

print(
    "Target leakage:",
    "False",
)

print(
    "Metadata consistency:",
    "PASS",
)

print(
    "Provenance integrity:",
    "PASS",
)

print(
    "R3_CELL_3_READY:",
    R3_CELL_3_READY,
)

assert R3_CELL_3_READY is True

print("=" * 80)
print("R3 CELL 3 — UNION INTEGRITY + PROVENANCE AUDIT: PASS")
print("=" * 80)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 3 — UNION INTEGRITY + PROVENANCE AUDIT

R3 Cell 0 dependency: PASS
R3 Cell 1 dependency: PASS
R3 Cell 2 dependency: PASS

UNION POPULATION
R1 source rows: 1,752,048
R2 source rows: 1,752,048
Expected overlap: 1,021,959
Observed union rows: 2,482,137
Expected union rows: 2,482,137

CANONICAL CANDIDATE IDENTITY
Identity columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid']
Duplicate identity rows: 0
Candidate identity: PASS

R3 SCHEMA
Required columns: 20
Missing columns: []
Schema contract: PASS

RESPONSE COVERAGE
R3 responses: 35,072
Expected responses: 35,072
R1/R3 response identity: MATCH
R2/R3 response identity: MATCH
Response coverage: PASS

SESSION COVERAGE
R3 sessions: 22,821
Expected sessions: 22,821
Session coverage: PASS

RESPONSE → SESSION CONSISTENCY
Responses mapped to >1 session: 0
Cross-session contamination: PASS

PROVENANCE
Sparse-only: 730,089
Dense-only: 730,089
Both: 1,021,959
Union total: 2,482,137
Proven

In [13]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 4 — UNION QUALITY DIAGNOSTICS
# ==============================================================================

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 4 — UNION QUALITY DIAGNOSTICS")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R3_CELL_0_READY is True, (
    "R3 Cell 0 must pass."
)

assert R3_CELL_1_READY is True, (
    "R3 Cell 1 must pass."
)

assert R3_CELL_2_READY is True, (
    "R3 Cell 2 must pass."
)

assert R3_CELL_3_READY is True, (
    "R3 Cell 3 must pass."
)


print("\nR3 Cell 0 dependency: PASS")
print("R3 Cell 1 dependency: PASS")
print("R3 Cell 2 dependency: PASS")
print("R3 Cell 3 dependency: PASS")


# ==============================================================================
# 2. SOURCE CONTRACT
# ==============================================================================

assert isinstance(
    r3_union,
    pd.DataFrame,
), "r3_union must be a pandas DataFrame."


REQUIRED_COLUMNS = {
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "sparse_score",
    "dense_score",
    "selected_sparse",
    "selected_dense",
}


missing_columns = (
    REQUIRED_COLUMNS
    -
    set(r3_union.columns)
)


assert not missing_columns, (
    "R3 union is missing required columns: "
    + repr(
        sorted(missing_columns)
    )
)


print("\nUnion schema contract: PASS")


# ==============================================================================
# 3. DIAGNOSTIC PATHS
# ==============================================================================

R3_DIAGNOSTIC_ROOT = (
    R3_ROOT
    / "diagnostics"
)

R3_DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


R3_UNION_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_union_diagnostics.parquet"
)

R3_RESPONSE_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_response_diagnostics.parquet"
)

R3_ROLE_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_role_diagnostics.parquet"
)

R3_RANK_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_rank_diagnostics.parquet"
)

R3_OVERLAP_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_overlap_diagnostics.json"
)

R3_DIAGNOSTICS_MANIFEST_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_diagnostics_manifest.json"
)


# ==============================================================================
# 4. BASIC UNION SUMMARY
# ==============================================================================

union_rows = len(r3_union)

union_responses = (
    r3_union[
        "response_id"
    ].nunique()
)

union_sessions = (
    r3_union[
        "session_id"
    ].nunique()
)

union_objectives = (
    r3_union[
        "objective_uid"
    ].nunique()
)

union_folds = sorted(
    r3_union[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert union_rows == 2_482_137, (
    f"Unexpected R3 union rows: {union_rows:,}"
)

assert union_responses == 35_072, (
    f"Unexpected response count: {union_responses:,}"
)

assert union_sessions == 22_821, (
    f"Unexpected session count: {union_sessions:,}"
)

assert union_objectives == 398, (
    f"Unexpected objective count: {union_objectives:,}"
)

assert union_folds == [0, 1, 2, 3, 4], (
    f"Unexpected folds: {union_folds}"
)


print("\n" + "=" * 80)
print("UNION SUMMARY")
print("=" * 80)

print(
    "Union candidates:",
    f"{union_rows:,}",
)

print(
    "Responses:",
    f"{union_responses:,}",
)

print(
    "Sessions:",
    f"{union_sessions:,}",
)

print(
    "Objectives:",
    f"{union_objectives:,}",
)

print(
    "Folds:",
    union_folds,
)


# ==============================================================================
# 5. PROVENANCE
# ==============================================================================

sparse_mask = (
    r3_union[
        "selected_sparse"
    ]
    .fillna(False)
    .astype(bool)
)

dense_mask = (
    r3_union[
        "selected_dense"
    ]
    .fillna(False)
    .astype(bool)
)


sparse_only_mask = (
    sparse_mask
    &
    ~dense_mask
)

dense_only_mask = (
    ~sparse_mask
    &
    dense_mask
)

both_mask = (
    sparse_mask
    &
    dense_mask
)


sparse_only_count = int(
    sparse_only_mask.sum()
)

dense_only_count = int(
    dense_only_mask.sum()
)

both_count = int(
    both_mask.sum()
)


assert (
    sparse_only_count
    +
    dense_only_count
    +
    both_count
    ==
    union_rows
), "Provenance accounting failed."


print("\n" + "=" * 80)
print("PROVENANCE DISTRIBUTION")
print("=" * 80)

print(
    "Sparse-only:",
    f"{sparse_only_count:,}",
)

print(
    "Dense-only:",
    f"{dense_only_count:,}",
)

print(
    "Both:",
    f"{both_count:,}",
)

print(
    "Provenance accounting: PASS"
)


# ==============================================================================
# 6. RESPONSE-LEVEL COVERAGE
# ==============================================================================

# ==============================================================================
# R3 CELL 4 — RESPONSE-LEVEL FORENSIC DIAGNOSTIC
# NO ASSERTIONS — DO NOT MODIFY R3 UNION
# ==============================================================================

print("\n" + "=" * 80)
print("R3 RESPONSE-LEVEL FORENSIC DIAGNOSTIC")
print("=" * 80)


# ------------------------------------------------------------------------------
# 1. BUILD R3 RESPONSE COUNTS
# ------------------------------------------------------------------------------

r3_response_diag = (
    r3_union
    .groupby(
        "response_id",
        sort=True,
    )
    .agg(
        r3_union_rows=(
            "turn_uid",
            "size",
        ),
        r3_sparse_selected=(
            "selected_sparse",
            "sum",
        ),
        r3_dense_selected=(
            "selected_dense",
            "sum",
        ),
        session_id=(
            "session_id",
            "first",
        ),
        objective_count=(
            "objective_uid",
            "nunique",
        ),
    )
    .reset_index()
)


print(
    "R3 responses:",
    f"{len(r3_response_diag):,}",
)


# ------------------------------------------------------------------------------
# 2. BUILD SOURCE RESPONSE COUNTS
# ------------------------------------------------------------------------------

r1_response_diag = (
    r1_union
    .groupby(
        "response_id",
        sort=True,
    )
    .agg(
        r1_rows=(
            "turn_uid",
            "size",
        ),
        r1_unique_turns=(
            "turn_uid",
            "nunique",
        ),
    )
    .reset_index()
)


r2_response_diag = (
    r2_union
    .groupby(
        "response_id",
        sort=True,
    )
    .agg(
        r2_rows=(
            "turn_uid",
            "size",
        ),
        r2_unique_turns=(
            "turn_uid",
            "nunique",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------------------------
# 3. MERGE SOURCE + R3 COUNTS
# ------------------------------------------------------------------------------

r3_forensic = (
    r1_response_diag
    .merge(
        r2_response_diag,
        on="response_id",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        r3_response_diag,
        on="response_id",
        how="outer",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------------------------
# 4. NULL / MISSING SOURCE CHECK
# ------------------------------------------------------------------------------

count_columns = [
    "r1_rows",
    "r1_unique_turns",
    "r2_rows",
    "r2_unique_turns",
    "r3_union_rows",
    "r3_sparse_selected",
    "r3_dense_selected",
]


for column in count_columns:
    r3_forensic[column] = (
        r3_forensic[column]
        .fillna(0)
        .astype(int)
    )


r3_forensic[
    "r1_missing"
] = (
    r3_forensic[
        "r1_rows"
    ]
    ==
    0
)


r3_forensic[
    "r2_missing"
] = (
    r3_forensic[
        "r2_rows"
    ]
    ==
    0
)


r3_forensic[
    "r3_missing"
] = (
    r3_forensic[
        "r3_union_rows"
    ]
    ==
    0
)


# ------------------------------------------------------------------------------
# 5. DEVIATION FLAGS
# ------------------------------------------------------------------------------

r3_forensic[
    "r1_not_50"
] = (
    r3_forensic[
        "r1_rows"
    ]
    !=
    50
)


r3_forensic[
    "r2_not_50"
] = (
    r3_forensic[
        "r2_rows"
    ]
    !=
    50
)


r3_forensic[
    "r3_below_50"
] = (
    r3_forensic[
        "r3_union_rows"
    ]
    <
    50
)


r3_forensic[
    "r3_above_100"
] = (
    r3_forensic[
        "r3_union_rows"
    ]
    >
    100
)


r3_forensic[
    "r3_sparse_not_50"
] = (
    r3_forensic[
        "r3_sparse_selected"
    ]
    !=
    50
)


r3_forensic[
    "r3_dense_not_50"
] = (
    r3_forensic[
        "r3_dense_selected"
    ]
    !=
    50
)


# ------------------------------------------------------------------------------
# 6. EXPECTED UNION FROM SOURCE COUNTS
# ------------------------------------------------------------------------------

# This is only an arithmetic diagnostic.
#
# We cannot derive exact unique union from counts alone because R1/R2
# candidate identity overlap varies by response.
#
# But:
#
#     lower bound = max(R1 rows, R2 rows)
#     upper bound = R1 rows + R2 rows
#
# Therefore if R1=50 and R2=50:
#
#     R3 union must be between 50 and 100.
# ------------------------------------------------------------------------------

r3_forensic[
    "expected_lower_bound"
] = (
    r3_forensic[
        [
            "r1_rows",
            "r2_rows",
        ]
    ]
    .max(
        axis=1
    )
)


r3_forensic[
    "expected_upper_bound"
] = (
    r3_forensic[
        [
            "r1_rows",
            "r2_rows",
        ]
    ]
    .sum(
        axis=1
    )
)


r3_forensic[
    "below_source_lower_bound"
] = (
    r3_forensic[
        "r3_union_rows"
    ]
    <
    r3_forensic[
        "expected_lower_bound"
    ]
)


r3_forensic[
    "above_source_upper_bound"
] = (
    r3_forensic[
        "r3_union_rows"
    ]
    >
    r3_forensic[
        "expected_upper_bound"
    ]
)


# ------------------------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE POPULATION SUMMARY")
print("=" * 80)

print(
    "Responses:",
    f"{len(r3_forensic):,}",
)

print(
    "R1 rows:",
    f"{r3_forensic['r1_rows'].sum():,}",
)

print(
    "R2 rows:",
    f"{r3_forensic['r2_rows'].sum():,}",
)

print(
    "R3 union rows:",
    f"{r3_forensic['r3_union_rows'].sum():,}",
)


print("\n" + "=" * 80)
print("R1 RESPONSE COUNTS")
print("=" * 80)

print(
    r3_forensic[
        "r1_rows"
    ].value_counts(
        dropna=False
    )
    .sort_index()
    .to_string()
)


print("\n" + "=" * 80)
print("R2 RESPONSE COUNTS")
print("=" * 80)

print(
    r3_forensic[
        "r2_rows"
    ].value_counts(
        dropna=False
    )
    .sort_index()
    .to_string()
)


print("\n" + "=" * 80)
print("R3 UNION RESPONSE COUNTS")
print("=" * 80)

print(
    r3_forensic[
        "r3_union_rows"
    ].value_counts(
        dropna=False
    )
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------------------------
# 8. CRITICAL DEVIATIONS
# ------------------------------------------------------------------------------

r1_bad = r3_forensic[
    r3_forensic[
        "r1_not_50"
    ]
].copy()


r2_bad = r3_forensic[
    r3_forensic[
        "r2_not_50"
    ]
].copy()


r3_bad = r3_forensic[
    r3_forensic[
        "r3_below_50"
    ]
].copy()


r3_sparse_bad = r3_forensic[
    r3_forensic[
        "r3_sparse_not_50"
    ]
].copy()


r3_dense_bad = r3_forensic[
    r3_forensic[
        "r3_dense_not_50"
    ]
].copy()


print("\n" + "=" * 80)
print("CRITICAL DEVIATION COUNTS")
print("=" * 80)

print(
    "R1 responses != 50:",
    f"{len(r1_bad):,}",
)

print(
    "R2 responses != 50:",
    f"{len(r2_bad):,}",
)

print(
    "R3 responses < 50:",
    f"{len(r3_bad):,}",
)

print(
    "R3 sparse_selected != 50:",
    f"{len(r3_sparse_bad):,}",
)

print(
    "R3 dense_selected != 50:",
    f"{len(r3_dense_bad):,}",
)


# ------------------------------------------------------------------------------
# 9. SHOW ACTUAL FAILING RESPONSES
# ------------------------------------------------------------------------------

if len(r3_bad) > 0:

    print("\n" + "=" * 80)
    print("RESPONSES WITH R3 UNION < 50")
    print("=" * 80)

    print(
        r3_bad[
            [
                "response_id",
                "session_id",
                "r1_rows",
                "r1_unique_turns",
                "r2_rows",
                "r2_unique_turns",
                "r3_union_rows",
                "r3_sparse_selected",
                "r3_dense_selected",
                "expected_lower_bound",
                "expected_upper_bound",
            ]
        ]
        .sort_values(
            [
                "r3_union_rows",
                "response_id",
            ]
        )
        .head(50)
        .to_string(
            index=False
        )
    )


# ------------------------------------------------------------------------------
# 10. SOURCE-LOWER-BOUND VIOLATIONS
# ------------------------------------------------------------------------------

lower_bound_bad = r3_forensic[
    r3_forensic[
        "below_source_lower_bound"
    ]
].copy()


print("\n" + "=" * 80)
print("SOURCE LOWER-BOUND VIOLATIONS")
print("=" * 80)

print(
    "Responses violating max(R1,R2) lower bound:",
    f"{len(lower_bound_bad):,}",
)


if len(lower_bound_bad) > 0:

    print(
        lower_bound_bad[
            [
                "response_id",
                "r1_rows",
                "r2_rows",
                "r3_union_rows",
                "expected_lower_bound",
                "expected_upper_bound",
            ]
        ]
        .sort_values(
            [
                "r3_union_rows",
                "response_id",
            ]
        )
        .head(50)
        .to_string(
            index=False
        )
    )


# ------------------------------------------------------------------------------
# 11. SAVE FORENSIC DIAGNOSTIC
# ------------------------------------------------------------------------------

R3_FORENSIC_RESPONSE_DIAGNOSTICS_PATH = (
    R3_DIAGNOSTIC_ROOT
    / "r3_response_forensic_diagnostics.parquet"
)


r3_forensic.to_parquet(
    R3_FORENSIC_RESPONSE_DIAGNOSTICS_PATH,
    index=False,
    engine="pyarrow",
)


assert (
    R3_FORENSIC_RESPONSE_DIAGNOSTICS_PATH.exists()
)


print("\n" + "=" * 80)
print("FORENSIC DIAGNOSTIC ARTIFACT")
print("=" * 80)

print(
    "Path:",
    R3_FORENSIC_RESPONSE_DIAGNOSTICS_PATH,
)

print(
    "Rows:",
    f"{len(r3_forensic):,}",
)

print(
    "Serialization:",
    "PASS",
)


# ==============================================================================
# IMPORTANT:
# Do NOT set R3_CELL_4_READY here.
# This is an investigation checkpoint, not a PASS gate.
# ==============================================================================

print("\n" + "=" * 80)
print("R3 CELL 4 FORENSIC CHECKPOINT")
print("=" * 80)

print(
    "R3 union modified:",
    "NO",
)

print(
    "R1/R2 modified:",
    "NO",
)

print(
    "Retrieval recomputed:",
    "NO",
)

print(
    "Freeze performed:",
    "NO",
)

print(
    "Diagnostic artifact written:",
    "YES",
)

print(
    "R3_CELL_4_READY:",
    globals().get(
        "R3_CELL_4_READY",
        False,
    ),
)

print("=" * 80)


# ==============================================================================
# 7. ROLE DISTRIBUTION
# ==============================================================================

role_diag = (
    r3_union
    .groupby(
        "role",
        sort=True,
    )
    .agg(
        rows=(
            "turn_uid",
            "size",
        ),
        responses=(
            "response_id",
            "nunique",
        ),
        sessions=(
            "session_id",
            "nunique",
        ),
        mean_turn_index=(
            "turn_index",
            "mean",
        ),
        median_turn_index=(
            "turn_index",
            "median",
        ),
        min_turn_index=(
            "turn_index",
            "min",
        ),
        max_turn_index=(
            "turn_index",
            "max",
        ),
    )
    .reset_index()
)


role_diag[
    "share"
] = (
    role_diag["rows"]
    /
    union_rows
)


assert (
    role_diag["rows"].sum()
    ==
    union_rows
)


print("\n" + "=" * 80)
print("ROLE DISTRIBUTION")
print("=" * 80)

print(
    role_diag.to_string(
        index=False
    )
)


# ==============================================================================
# 8. RANK-WISE DIAGNOSTICS
# ==============================================================================

rank_frames = []


# ------------------------------------------------------------------------------
# Sparse
# ------------------------------------------------------------------------------

if "sparse_rank" in r3_union.columns:

    sparse_rank_diag = (
        r3_union[
            sparse_mask
            &
            r3_union[
                "sparse_rank"
            ].notna()
        ]
        .groupby(
            "sparse_rank",
            sort=True,
        )
        .agg(
            candidates=(
                "turn_uid",
                "size",
            ),
            responses=(
                "response_id",
                "nunique",
            ),
            mean_score=(
                "sparse_score",
                "mean",
            ),
            median_score=(
                "sparse_score",
                "median",
            ),
            min_score=(
                "sparse_score",
                "min",
            ),
            max_score=(
                "sparse_score",
                "max",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "sparse_rank": "rank",
            }
        )
    )

    sparse_rank_diag[
        "retriever"
    ] = "sparse"

    rank_frames.append(
        sparse_rank_diag
    )

else:

    print(
        "\nSparse rank column not present; "
        "sparse rank diagnostics skipped."
    )


# ------------------------------------------------------------------------------
# Dense
# ------------------------------------------------------------------------------

if "dense_rank" in r3_union.columns:

    dense_rank_diag = (
        r3_union[
            dense_mask
            &
            r3_union[
                "dense_rank"
            ].notna()
        ]
        .groupby(
            "dense_rank",
            sort=True,
        )
        .agg(
            candidates=(
                "turn_uid",
                "size",
            ),
            responses=(
                "response_id",
                "nunique",
            ),
            mean_score=(
                "dense_score",
                "mean",
            ),
            median_score=(
                "dense_score",
                "median",
            ),
            min_score=(
                "dense_score",
                "min",
            ),
            max_score=(
                "dense_score",
                "max",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "dense_rank": "rank",
            }
        )
    )

    dense_rank_diag[
        "retriever"
    ] = "dense"

    rank_frames.append(
        dense_rank_diag
    )

else:

    print(
        "\nDense rank column not present; "
        "dense rank diagnostics skipped."
    )


assert len(rank_frames) > 0, (
    "No rank diagnostics could be constructed."
)


rank_diag = pd.concat(
    rank_frames,
    axis=0,
    ignore_index=True,
)


print("\n" + "=" * 80)
print("RANK-WISE DIAGNOSTICS")
print("=" * 80)

print(
    rank_diag[
        [
            "retriever",
            "rank",
            "candidates",
            "responses",
            "mean_score",
            "median_score",
            "min_score",
            "max_score",
        ]
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 9. OVERLAP METRICS
# ==============================================================================

r1_count = 1_752_048
r2_count = 1_752_048


raw_combined_rows = (
    r1_count
    +
    r2_count
)


expected_union_rows = (
    raw_combined_rows
    -
    both_count
)


assert (
    expected_union_rows
    ==
    union_rows
), (
    "Union population does not reconcile "
    "with provenance overlap."
)


overlap_share = (
    both_count
    /
    r1_count
)


union_multiplier = (
    union_rows
    /
    r1_count
)


overlap_summary = {
    "r1_candidates": r1_count,
    "r2_candidates": r2_count,
    "raw_combined_rows": raw_combined_rows,
    "overlap_rows": both_count,
    "sparse_only_rows": sparse_only_count,
    "dense_only_rows": dense_only_count,
    "unique_union_rows": union_rows,
    "overlap_share_of_single_retriever": float(
        overlap_share
    ),
    "union_multiplier": float(
        union_multiplier
    ),
}


with open(
    R3_OVERLAP_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        overlap_summary,
        handle,
        indent=2,
    )


print("\n" + "=" * 80)
print("OVERLAP")
print("=" * 80)

print(
    "Raw combined:",
    f"{raw_combined_rows:,}",
)

print(
    "Overlap:",
    f"{both_count:,}",
)

print(
    "Unique union:",
    f"{union_rows:,}",
)

print(
    "Overlap share:",
    f"{overlap_share:.6f}",
)

print(
    "Union multiplier:",
    f"{union_multiplier:.6f}",
)

print(
    "Overlap accounting: PASS"
)


# ==============================================================================
# 10. SCORE DISTRIBUTIONS
# ==============================================================================

score_rows = []


for column in [
    "sparse_score",
    "dense_score",
    "word_score",
    "char_score",
    "math_score",
]:

    if column not in r3_union.columns:
        continue

    values = (
        r3_union[
            column
        ]
        .dropna()
        .astype(float)
    )

    if len(values) == 0:
        continue

    score_rows.append(
        {
            "metric": column,
            "count": int(
                len(values)
            ),
            "mean": float(
                values.mean()
            ),
            "median": float(
                values.median()
            ),
            "std": float(
                values.std()
            ),
            "min": float(
                values.min()
            ),
            "max": float(
                values.max()
            ),
            "p01": float(
                values.quantile(
                    0.01
                )
            ),
            "p05": float(
                values.quantile(
                    0.05
                )
            ),
            "p95": float(
                values.quantile(
                    0.95
                )
            ),
            "p99": float(
                values.quantile(
                    0.99
                )
            ),
        }
    )


score_summary_df = pd.DataFrame(
    score_rows
)


print("\n" + "=" * 80)
print("SCORE DISTRIBUTIONS")
print("=" * 80)

print(
    score_summary_df.to_string(
        index=False
    )
)


# ==============================================================================
# 11. COMPACT UNION SUMMARY
# ==============================================================================

union_summary_df = pd.DataFrame(
    {
        "metric": [
            "union_rows",
            "responses",
            "sessions",
            "objectives",
            "sparse_only",
            "dense_only",
            "both_sources",
            "overlap_share",
            "union_multiplier",
        ],
        "value": [
            union_rows,
            union_responses,
            union_sessions,
            union_objectives,
            sparse_only_count,
            dense_only_count,
            both_count,
            overlap_share,
            union_multiplier,
        ],
    }
)


# ==============================================================================
# 12. WRITE DIAGNOSTIC ARTIFACTS
# ==============================================================================

print("\n" + "=" * 80)
print("WRITE DIAGNOSTIC ARTIFACTS")
print("=" * 80)


union_summary_df.to_parquet(
    R3_UNION_DIAGNOSTICS_PATH,
    index=False,
    engine="pyarrow",
)


response_diag.to_parquet(
    R3_RESPONSE_DIAGNOSTICS_PATH,
    index=False,
    engine="pyarrow",
)


role_diag.to_parquet(
    R3_ROLE_DIAGNOSTICS_PATH,
    index=False,
    engine="pyarrow",
)


rank_diag.to_parquet(
    R3_RANK_DIAGNOSTICS_PATH,
    index=False,
    engine="pyarrow",
)


print(
    "Union diagnostics: PASS"
)

print(
    "Response diagnostics: PASS"
)

print(
    "Role diagnostics: PASS"
)

print(
    "Rank diagnostics: PASS"
)

print(
    "Overlap diagnostics: PASS"
)


# ==============================================================================
# 13. DIAGNOSTICS MANIFEST
# ==============================================================================

R3_DIAGNOSTIC_MANIFEST = {
    "artifact": "r3_union_diagnostics",
    "status": "COMPLETE",
    "union_rows": int(
        union_rows
    ),
    "responses": int(
        union_responses
    ),
    "sessions": int(
        union_sessions
    ),
    "objectives": int(
        union_objectives
    ),
    "folds": union_folds,
    "r1_candidates": r1_count,
    "r2_candidates": r2_count,
    "overlap_rows": int(
        both_count
    ),
    "sparse_only_rows": int(
        sparse_only_count
    ),
    "dense_only_rows": int(
        dense_only_count
    ),
    "diagnostic_files": [
        str(
            R3_UNION_DIAGNOSTICS_PATH
        ),
        str(
            R3_RESPONSE_DIAGNOSTICS_PATH
        ),
        str(
            R3_ROLE_DIAGNOSTICS_PATH
        ),
        str(
            R3_RANK_DIAGNOSTICS_PATH
        ),
        str(
            R3_OVERLAP_DIAGNOSTICS_PATH
        ),
    ],
}


with open(
    R3_DIAGNOSTICS_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        R3_DIAGNOSTIC_MANIFEST,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ==============================================================================
# 14. SERIALIZATION VERIFICATION
# ==============================================================================

for artifact_path in [
    R3_UNION_DIAGNOSTICS_PATH,
    R3_RESPONSE_DIAGNOSTICS_PATH,
    R3_ROLE_DIAGNOSTICS_PATH,
    R3_RANK_DIAGNOSTICS_PATH,
    R3_OVERLAP_DIAGNOSTICS_PATH,
    R3_DIAGNOSTICS_MANIFEST_PATH,
]:

    assert Path(
        artifact_path
    ).exists(), (
        f"Missing diagnostic artifact: "
        f"{artifact_path}"
    )


with open(
    R3_DIAGNOSTICS_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    loaded_manifest = json.load(
        handle
    )


assert (
    loaded_manifest[
        "status"
    ]
    ==
    "COMPLETE"
)


print(
    "Diagnostic serialization: PASS"
)


# ==============================================================================
# 15. FINAL CELL GATE
# ==============================================================================

R3_CELL_4_READY = True


print("\n" + "=" * 80)
print("R3 CELL 4 STATUS")
print("=" * 80)

print(
    "Union candidates:",
    f"{union_rows:,}",
)

print(
    "Responses:",
    f"{union_responses:,}",
)

print(
    "Sessions:",
    f"{union_sessions:,}",
)

print(
    "Sparse-only:",
    f"{sparse_only_count:,}",
)

print(
    "Dense-only:",
    f"{dense_only_count:,}",
)

print(
    "Both sources:",
    f"{both_count:,}",
)

print(
    "Overlap accounting:",
    "PASS",
)

print(
    "Response coverage:",
    "PASS",
)

print(
    "Diagnostic artifacts:",
    "PASS",
)

print(
    "R3_CELL_4_READY:",
    R3_CELL_4_READY,
)

assert R3_CELL_4_READY is True

print("=" * 80)
print(
    "R3 CELL 4 — UNION QUALITY DIAGNOSTICS: PASS"
)
print("=" * 80)


# ==============================================================================
# 16. MEMORY CLEANUP
# ==============================================================================

if "response_diag" in globals():
    del response_diag

if "role_diag" in globals():
    del role_diag

if "rank_diag" in globals():
    del rank_diag

if "sparse_rank_diag" in globals():
    del sparse_rank_diag

if "dense_rank_diag" in globals():
    del dense_rank_diag

if "score_summary_df" in globals():
    del score_summary_df

if "union_summary_df" in globals():
    del union_summary_df

gc.collect()

print(
    "\nR3 Cell 4 memory cleanup: PASS"
)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 4 — UNION QUALITY DIAGNOSTICS

R3 Cell 0 dependency: PASS
R3 Cell 1 dependency: PASS
R3 Cell 2 dependency: PASS
R3 Cell 3 dependency: PASS

Union schema contract: PASS

UNION SUMMARY
Union candidates: 2,482,137
Responses: 35,072
Sessions: 22,821
Objectives: 398
Folds: [0, 1, 2, 3, 4]

PROVENANCE DISTRIBUTION
Sparse-only: 730,089
Dense-only: 730,089
Both: 1,021,959
Provenance accounting: PASS

R3 RESPONSE-LEVEL FORENSIC DIAGNOSTIC
R3 responses: 35,072

SOURCE POPULATION SUMMARY
Responses: 35,072
R1 rows: 1,752,048
R2 rows: 1,752,048
R3 union rows: 2,482,137

R1 RESPONSE COUNTS
r1_rows
15        2
16        9
20        1
21        4
22        1
24        8
25        2
27        2
28        3
29        5
30        1
31        1
32        7
33        1
34        3
35        4
36        1
37        2
38        1
39        1
40        3
42        9
43        6
44        1
46        2
47        5
49        1
50    34986

R2 RESPONSE COUNTS
r2_rows
15   

In [14]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 6 — FINAL R3 FREEZE + INTEGRITY MANIFEST
# ==============================================================================

import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 6 — FINAL R3 FREEZE + INTEGRITY MANIFEST")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATE
# ==============================================================================

assert R3_CELL_0_READY is True, (
    "R3 Cell 0 must pass."
)

assert R3_CELL_1_READY is True, (
    "R3 Cell 1 must pass."
)

assert R3_CELL_2_READY is True, (
    "R3 Cell 2 must pass."
)

assert R3_CELL_3_READY is True, (
    "R3 Cell 3 must pass."
)

assert R3_CELL_4_READY is True, (
    "R3 quality diagnostics must pass before freeze."
)


print("\nR3 Cell 0 dependency : PASS")
print("R3 Cell 1 dependency : PASS")
print("R3 Cell 2 dependency : PASS")
print("R3 Cell 3 dependency : PASS")
print("R3 diagnostics       : PASS")


# ==============================================================================
# 1. SOURCE UNION CONTRACT
# ==============================================================================

assert isinstance(
    r3_union,
    pd.DataFrame,
), "r3_union must be a pandas DataFrame."


REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "sparse_score",
    "dense_score",
    "selected_sparse",
    "selected_dense",
]


missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in r3_union.columns
]


assert not missing_columns, (
    "R3 union is missing required columns: "
    + repr(missing_columns)
)


print(
    "\nR3 union schema contract: PASS"
)


# ==============================================================================
# 2. FREEZE PATHS
# ==============================================================================

R3_FREEZE_ROOT = (
    R3_ROOT
    / "frozen"
)

R3_FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


R3_CANDIDATE_FREEZE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

R3_FREEZE_MANIFEST_PATH = (
    R3_FREEZE_ROOT
    / "r3_cell6_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("R3 FREEZE PATHS")
print("=" * 80)

print(
    "Freeze root:",
    R3_FREEZE_ROOT,
)

print(
    "Candidates:",
    R3_CANDIDATE_FREEZE_PATH,
)

print(
    "Manifest:",
    R3_FREEZE_MANIFEST_PATH,
)


# ==============================================================================
# 3. HARD POPULATION CONTRACT
# ==============================================================================

EXPECTED_R1_ROWS = 1_752_048
EXPECTED_R2_ROWS = 1_752_048

EXPECTED_UNION_ROWS = 2_482_137
EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_OBJECTIVES = 398
EXPECTED_FOLDS = [0, 1, 2, 3, 4]


union_rows = len(
    r3_union
)

response_count = (
    r3_union[
        "response_id"
    ].nunique()
)

session_count = (
    r3_union[
        "session_id"
    ].nunique()
)

objective_count = (
    r3_union[
        "objective_uid"
    ].nunique()
)

folds = sorted(
    r3_union[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert union_rows == EXPECTED_UNION_ROWS, (
    f"Unexpected R3 union rows: "
    f"{union_rows:,}"
)

assert response_count == EXPECTED_RESPONSES, (
    f"Unexpected response count: "
    f"{response_count:,}"
)

assert session_count == EXPECTED_SESSIONS, (
    f"Unexpected session count: "
    f"{session_count:,}"
)

assert objective_count == EXPECTED_OBJECTIVES, (
    f"Unexpected objective count: "
    f"{objective_count:,}"
)

assert folds == EXPECTED_FOLDS, (
    f"Unexpected folds: {folds}"
)


print("\n" + "=" * 80)
print("POPULATION CONTRACT")
print("=" * 80)

print(
    "Union rows:",
    f"{union_rows:,}",
)

print(
    "Responses:",
    f"{response_count:,}",
)

print(
    "Sessions:",
    f"{session_count:,}",
)

print(
    "Objectives:",
    f"{objective_count:,}",
)

print(
    "Folds:",
    folds,
)

print(
    "Population contract: PASS"
)


# ==============================================================================
# 4. CANONICAL CANDIDATE IDENTITY AUDIT
# ==============================================================================

IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
]


duplicate_identity_count = int(
    r3_union.duplicated(
        subset=IDENTITY_COLUMNS
    ).sum()
)


assert duplicate_identity_count == 0, (
    "Duplicate R3 candidate identities detected: "
    f"{duplicate_identity_count:,}"
)


print("\n" + "=" * 80)
print("CANDIDATE IDENTITY")
print("=" * 80)

print(
    "Identity columns:",
    IDENTITY_COLUMNS,
)

print(
    "Duplicate identities:",
    duplicate_identity_count,
)

print(
    "Candidate identity: PASS"
)


# ==============================================================================
# 5. PROVENANCE CONTRACT
# ==============================================================================

sparse_mask = (
    r3_union[
        "selected_sparse"
    ]
    .fillna(False)
    .astype(bool)
)

dense_mask = (
    r3_union[
        "selected_dense"
    ]
    .fillna(False)
    .astype(bool)
)


sparse_only_count = int(
    (
        sparse_mask
        &
        ~dense_mask
    ).sum()
)

dense_only_count = int(
    (
        ~sparse_mask
        &
        dense_mask
    ).sum()
)

both_count = int(
    (
        sparse_mask
        &
        dense_mask
    ).sum()
)


assert (
    sparse_only_count
    +
    dense_only_count
    +
    both_count
    ==
    union_rows
)


assert (
    sparse_only_count
    ==
    730_089
)

assert (
    dense_only_count
    ==
    730_089
)

assert (
    both_count
    ==
    1_021_959
)


print("\n" + "=" * 80)
print("PROVENANCE CONTRACT")
print("=" * 80)

print(
    "Sparse-only:",
    f"{sparse_only_count:,}",
)

print(
    "Dense-only:",
    f"{dense_only_count:,}",
)

print(
    "Both:",
    f"{both_count:,}",
)

print(
    "Provenance accounting: PASS"
)


# ==============================================================================
# 6. TARGET / LABEL ISOLATION
# ==============================================================================

TARGET_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
}


target_columns_present = (
    TARGET_COLUMNS
    &
    set(r3_union.columns)
)


assert not target_columns_present, (
    "Target/label column leaked into R3 candidate artifact: "
    + repr(
        sorted(target_columns_present)
    )
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target columns present:",
    sorted(
        target_columns_present
    ),
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation: PASS"
)


# ==============================================================================
# 7. SAME-SESSION CONTRACT
# ==============================================================================

# R3 candidate identity must remain inside the response's session.
#
# Build response → session mapping from the R0 retrieval query artifact
# if available. Otherwise use the already validated R3 response/session
# relationship as the local contract.

if (
    "retrieval_queries"
    in globals()
):

    rq = retrieval_queries[
        [
            "response_id",
            "session_id",
        ]
    ].drop_duplicates(
        "response_id"
    )

    response_session_map = dict(
        zip(
            rq[
                "response_id"
            ].astype(str),
            rq[
                "session_id"
            ].astype(str),
        )
    )

    expected_sessions = (
        r3_union[
            "response_id"
        ]
        .astype(str)
        .map(
            response_session_map
        )
    )

    session_mismatch_count = int(
        (
            r3_union[
                "session_id"
            ]
            .astype(str)
            !=
            expected_sessions
        ).sum()
    )

    assert (
        session_mismatch_count
        ==
        0
    ), (
        "Cross-session candidates detected: "
        f"{session_mismatch_count:,}"
    )

else:

    session_mismatch_count = 0

    print(
        "R0 retrieval_queries not in memory; "
        "using prior R3 session-boundary certification."
    )


print("\n" + "=" * 80)
print("SESSION BOUNDARY")
print("=" * 80)

print(
    "Cross-session mismatches:",
    session_mismatch_count,
)

print(
    "Same-session contract: PASS"
)


# ==============================================================================
# 8. SHA256 HELPER
# ==============================================================================

def r3_sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


# ==============================================================================
# 9. WRITE FROZEN PARQUET
# ==============================================================================

print("\n" + "=" * 80)
print("WRITE FROZEN CANDIDATE ARTIFACT")
print("=" * 80)


tmp_candidate_path = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet.tmp"
)


if tmp_candidate_path.exists():
    tmp_candidate_path.unlink()


r3_union.to_parquet(
    tmp_candidate_path,
    index=False,
    engine="pyarrow",
)


assert tmp_candidate_path.exists()


# Atomic replacement.
tmp_candidate_path.replace(
    R3_CANDIDATE_FREEZE_PATH
)


assert (
    R3_CANDIDATE_FREEZE_PATH.exists()
)


print(
    "Candidate parquet written: PASS"
)


# ==============================================================================
# 10. RELOAD FROZEN CANDIDATE
# ==============================================================================

frozen_r3 = pd.read_parquet(
    R3_CANDIDATE_FREEZE_PATH,
    engine="pyarrow",
)


assert len(
    frozen_r3
) == union_rows


assert list(
    frozen_r3.columns
) == list(
    r3_union.columns
)


assert (
    frozen_r3[
        IDENTITY_COLUMNS
    ].duplicated().sum()
    ==
    0
)


print(
    "Frozen candidate reload: PASS"
)


# ==============================================================================
# 11. RELOADED POPULATION VERIFICATION
# ==============================================================================

assert (
    frozen_r3[
        "response_id"
    ].nunique()
    ==
    EXPECTED_RESPONSES
)

assert (
    frozen_r3[
        "session_id"
    ].nunique()
    ==
    EXPECTED_SESSIONS
)

assert (
    frozen_r3[
        "objective_uid"
    ].nunique()
    ==
    EXPECTED_OBJECTIVES
)

assert (
    sorted(
        frozen_r3[
            "fold"
        ]
        .astype(int)
        .unique()
        .tolist()
    )
    ==
    EXPECTED_FOLDS
)


frozen_sparse_mask = (
    frozen_r3[
        "selected_sparse"
    ]
    .fillna(False)
    .astype(bool)
)

frozen_dense_mask = (
    frozen_r3[
        "selected_dense"
    ]
    .fillna(False)
    .astype(bool)
)


assert int(
    (
        frozen_sparse_mask
        &
        ~frozen_dense_mask
    ).sum()
) == sparse_only_count


assert int(
    (
        ~frozen_sparse_mask
        &
        frozen_dense_mask
    ).sum()
) == dense_only_count


assert int(
    (
        frozen_sparse_mask
        &
        frozen_dense_mask
    ).sum()
) == both_count


print(
    "Frozen population verification: PASS"
)


# ==============================================================================
# 12. HASH FROZEN ARTIFACT
# ==============================================================================

candidate_sha256 = (
    r3_sha256_file(
        R3_CANDIDATE_FREEZE_PATH
    )
)


candidate_size_bytes = (
    R3_CANDIDATE_FREEZE_PATH.stat().st_size
)


print("\n" + "=" * 80)
print("CANDIDATE INTEGRITY")
print("=" * 80)

print(
    "Size:",
    f"{candidate_size_bytes:,}",
    "bytes",
)

print(
    "SHA256:",
    candidate_sha256,
)


# ==============================================================================
# 13. SOURCE ARTIFACT IDENTITY
# ==============================================================================

R1_FROZEN_CANDIDATES = (
    R1_ROOT
    / "frozen"
    / "r1_sparse_candidates.parquet"
)

R1_FREEZE_MANIFEST = (
    R1_ROOT
    / "frozen"
    / "r1_cell6_freeze_manifest.json"
)

R2_FROZEN_CANDIDATES = (
    R2_ROOT
    / "frozen"
    / "r2_dense_candidates.parquet"
)

R2_FREEZE_MANIFEST = (
    R2_ROOT
    / "frozen"
    / "r2_dense_freeze_manifest.json"
)


assert R1_FROZEN_CANDIDATES.exists(), (
    f"Missing frozen R1 artifact:\n"
    f"{R1_FROZEN_CANDIDATES}"
)

assert R1_FREEZE_MANIFEST.exists(), (
    f"Missing R1 freeze manifest:\n"
    f"{R1_FREEZE_MANIFEST}"
)

assert R2_FROZEN_CANDIDATES.exists(), (
    f"Missing frozen R2 artifact:\n"
    f"{R2_FROZEN_CANDIDATES}"
)

assert R2_FREEZE_MANIFEST.exists(), (
    f"Missing R2 freeze manifest:\n"
    f"{R2_FREEZE_MANIFEST}"
)


r1_sha256 = (
    r3_sha256_file(
        R1_FROZEN_CANDIDATES
    )
)

r2_sha256 = (
    r3_sha256_file(
        R2_FROZEN_CANDIDATES
    )
)


r1_manifest_sha256 = (
    r3_sha256_file(
        R1_FREEZE_MANIFEST
    )
)

r2_manifest_sha256 = (
    r3_sha256_file(
        R2_FREEZE_MANIFEST
    )
)


print("\n" + "=" * 80)
print("UPSTREAM ARTIFACT IDENTITY")
print("=" * 80)

print(
    "R1 candidate SHA256:",
    r1_sha256,
)

print(
    "R2 candidate SHA256:",
    r2_sha256,
)

print(
    "R1 manifest SHA256:",
    r1_manifest_sha256,
)

print(
    "R2 manifest SHA256:",
    r2_manifest_sha256,
)


# ==============================================================================
# 14. MANIFEST
# ==============================================================================

created_at = (
    datetime.now(
        timezone.utc
    )
    .isoformat()
)


R3_FREEZE_MANIFEST = {
    "artifact": "r3_candidate_union",
    "status": "FROZEN",
    "version": "R3_CELL6_FREEZE_1",

    "candidate_artifact": (
        str(
            R3_CANDIDATE_FREEZE_PATH.resolve()
        )
    ),

    "candidate_sha256": candidate_sha256,

    "candidate_size_bytes": int(
        candidate_size_bytes
    ),

    "rows": int(
        union_rows
    ),

    "responses": int(
        response_count
    ),

    "sessions": int(
        session_count
    ),

    "objectives": int(
        objective_count
    ),

    "folds": folds,

    "identity_columns": IDENTITY_COLUMNS,

    "schema_columns": [
        str(column)
        for column in frozen_r3.columns
    ],

    "provenance": {
        "sparse_only": int(
            sparse_only_count
        ),
        "dense_only": int(
            dense_only_count
        ),
        "both_sources": int(
            both_count
        ),
    },

    "source_artifacts": {
        "r1_candidates": {
            "path": str(
                R1_FROZEN_CANDIDATES.resolve()
            ),
            "sha256": r1_sha256,
        },
        "r1_manifest": {
            "path": str(
                R1_FREEZE_MANIFEST.resolve()
            ),
            "sha256": r1_manifest_sha256,
        },
        "r2_candidates": {
            "path": str(
                R2_FROZEN_CANDIDATES.resolve()
            ),
            "sha256": r2_sha256,
        },
        "r2_manifest": {
            "path": str(
                R2_FREEZE_MANIFEST.resolve()
            ),
            "sha256": r2_manifest_sha256,
        },
    },

    "retrieval_contract": {
        "candidate_scope": "same_session_only",
        "r1_top_k": 50,
        "r2_top_k": 50,
        "target_used": False,
        "cross_session_contamination": False,
        "deduplicated": True,
    },

    "integrity": {
        "duplicate_identity_rows": 0,
        "target_columns_present": [],
        "session_mismatch_rows": int(
            session_mismatch_count
        ),
    },

    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },

    "created_at_utc": created_at,
}


# ==============================================================================
# 15. WRITE MANIFEST
# ==============================================================================

tmp_manifest_path = (
    R3_FREEZE_ROOT
    / "r3_cell6_freeze_manifest.json.tmp"
)


if tmp_manifest_path.exists():
    tmp_manifest_path.unlink()


with open(
    tmp_manifest_path,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        R3_FREEZE_MANIFEST,
        handle,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )


tmp_manifest_path.replace(
    R3_FREEZE_MANIFEST_PATH
)


assert (
    R3_FREEZE_MANIFEST_PATH.exists()
)


print(
    "Freeze manifest written: PASS"
)


# ==============================================================================
# 16. MANIFEST RELOAD + SELF-CHECK
# ==============================================================================

with open(
    R3_FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    loaded_manifest = json.load(
        handle
    )


assert (
    loaded_manifest[
        "status"
    ]
    ==
    "FROZEN"
)


assert (
    loaded_manifest[
        "rows"
    ]
    ==
    union_rows
)


assert (
    loaded_manifest[
        "responses"
    ]
    ==
    response_count
)


assert (
    loaded_manifest[
        "sessions"
    ]
    ==
    session_count
)


assert (
    loaded_manifest[
        "objectives"
    ]
    ==
    objective_count
)


assert (
    loaded_manifest[
        "candidate_sha256"
    ]
    ==
    candidate_sha256
)


assert (
    loaded_manifest[
        "provenance"
    ][
        "sparse_only"
    ]
    ==
    sparse_only_count
)


assert (
    loaded_manifest[
        "provenance"
    ][
        "dense_only"
    ]
    ==
    dense_only_count
)


assert (
    loaded_manifest[
        "provenance"
    ][
        "both_sources"
    ]
    ==
    both_count
)


print(
    "Manifest self-verification: PASS"
)


# ==============================================================================
# 17. FINAL FREEZE FLAG
# ==============================================================================

R3_CANDIDATE_UNION_FROZEN = True
R3_CELL_6_READY = True


print("\n" + "=" * 80)
print("R3 CELL 6 STATUS")
print("=" * 80)

print(
    "Frozen candidate rows:",
    f"{union_rows:,}",
)

print(
    "Responses:",
    f"{response_count:,}",
)

print(
    "Sessions:",
    f"{session_count:,}",
)

print(
    "Objectives:",
    f"{objective_count:,}",
)

print(
    "Candidate artifact:",
    R3_CANDIDATE_FREEZE_PATH,
)

print(
    "Manifest:",
    R3_FREEZE_MANIFEST_PATH,
)

print(
    "SHA256:",
    candidate_sha256,
)

print(
    "Target used:",
    False,
)

print(
    "Cross-session contamination:",
    False,
)

print(
    "Duplicate identities:",
    0,
)

print(
    "R3_CANDIDATE_UNION_FROZEN:",
    R3_CANDIDATE_UNION_FROZEN,
)

print(
    "R3_CELL_6_READY:",
    R3_CELL_6_READY,
)


assert (
    R3_CANDIDATE_UNION_FROZEN
    is True
)

assert (
    R3_CELL_6_READY
    is True
)


print("=" * 80)
print(
    "R3 CELL 6 — FINAL CANDIDATE FREEZE: PASS"
)
print("=" * 80)


# ==============================================================================
# 18. MEMORY CLEANUP
# ==============================================================================

if "frozen_r3" in globals():
    del frozen_r3

if "rq" in globals():
    del rq

if "expected_sessions" in globals():
    del expected_sessions

if "response_session_map" in globals():
    del response_session_map

gc.collect()

print(
    "\nR3 Cell 6 memory cleanup: PASS"
)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 6 — FINAL R3 FREEZE + INTEGRITY MANIFEST

R3 Cell 0 dependency : PASS
R3 Cell 1 dependency : PASS
R3 Cell 2 dependency : PASS
R3 Cell 3 dependency : PASS
R3 diagnostics       : PASS

R3 union schema contract: PASS

R3 FREEZE PATHS
Freeze root: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen
Candidates: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_candidate_union.parquet
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_cell6_freeze_manifest.json

POPULATION CONTRACT
Union rows: 2,482,137
Responses: 35,072
Sessions: 22,821
Objectives: 398
Folds: [0, 1, 2, 3, 4]
Population contract: PASS

CANDIDATE IDENTITY
Identity columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid']
Duplicate identities: 0
Candidate identity: PASS

PROVENANCE CONTRACT
Sparse-only: 730,089
Dense-only: 730,089
Both: 1,021

In [17]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION
# ==============================================================================

import gc
import hashlib
import json
from pathlib import Path

import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION")
print("=" * 80)


# ==============================================================================
# 1. FROZEN PATHS
# ==============================================================================

R3_FREEZE_ROOT = (
    R3_ROOT
    / "frozen"
)

R3_CANDIDATE_FREEZE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

R3_FREEZE_MANIFEST_PATH = (
    R3_FREEZE_ROOT
    / "r3_cell6_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("FROZEN ARTIFACT PATHS")
print("=" * 80)

print(
    "Frozen root:",
    R3_FREEZE_ROOT,
)

print(
    "Candidates:",
    R3_CANDIDATE_FREEZE_PATH,
)

print(
    "Manifest:",
    R3_FREEZE_MANIFEST_PATH,
)


# ==============================================================================
# 2. ARTIFACT EXISTENCE
# ==============================================================================

assert R3_FREEZE_ROOT.exists(), (
    f"Missing R3 frozen directory:\n"
    f"{R3_FREEZE_ROOT}"
)

assert R3_CANDIDATE_FREEZE_PATH.exists(), (
    f"Missing R3 frozen candidate artifact:\n"
    f"{R3_CANDIDATE_FREEZE_PATH}"
)

assert R3_FREEZE_MANIFEST_PATH.exists(), (
    f"Missing R3 freeze manifest:\n"
    f"{R3_FREEZE_MANIFEST_PATH}"
)


print("\n" + "=" * 80)
print("ARTIFACT EXISTENCE")
print("=" * 80)

print(
    "Frozen directory:",
    True,
)

print(
    "Candidate parquet:",
    True,
)

print(
    "Freeze manifest:",
    True,
)


# ==============================================================================
# 3. LOAD MANIFEST
# ==============================================================================

with open(
    R3_FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    r3_freeze_manifest = json.load(
        handle
    )


assert isinstance(
    r3_freeze_manifest,
    dict,
)


assert (
    r3_freeze_manifest.get(
        "status"
    )
    ==
    "FROZEN"
)


assert (
    r3_freeze_manifest.get(
        "artifact"
    )
    ==
    "r3_candidate_union"
)


print("\n" + "=" * 80)
print("FREEZE MANIFEST")
print("=" * 80)

print(
    "JSON valid:",
    True,
)

print(
    "Status:",
    r3_freeze_manifest[
        "status"
    ],
)

print(
    "Artifact:",
    r3_freeze_manifest[
        "artifact"
    ],
)


# ==============================================================================
# 4. EXPECTED CONTRACT FROM MANIFEST
# ==============================================================================

EXPECTED_ROWS = int(
    r3_freeze_manifest[
        "rows"
    ]
)

EXPECTED_RESPONSES = int(
    r3_freeze_manifest[
        "responses"
    ]
)

EXPECTED_SESSIONS = int(
    r3_freeze_manifest[
        "sessions"
    ]
)

EXPECTED_OBJECTIVES = int(
    r3_freeze_manifest[
        "objectives"
    ]
)

EXPECTED_FOLDS = [
    int(x)
    for x in
    r3_freeze_manifest[
        "folds"
    ]
]


EXPECTED_SPARSE_ONLY = int(
    r3_freeze_manifest[
        "provenance"
    ][
        "sparse_only"
    ]
)

EXPECTED_DENSE_ONLY = int(
    r3_freeze_manifest[
        "provenance"
    ][
        "dense_only"
    ]
)

EXPECTED_BOTH = int(
    r3_freeze_manifest[
        "provenance"
    ][
        "both_sources"
    ]
)


EXPECTED_SHA256 = (
    r3_freeze_manifest[
        "candidate_sha256"
    ]
)


EXPECTED_COLUMNS = (
    r3_freeze_manifest[
        "schema_columns"
    ]
)


EXPECTED_IDENTITY_COLUMNS = (
    r3_freeze_manifest[
        "identity_columns"
    ]
)


# ==============================================================================
# 5. SHA256
# ==============================================================================

def r3_cell7_sha256(
    path,
    chunk_size=1024 * 1024,
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


print("\n" + "=" * 80)
print("SHA256 INTEGRITY")
print("=" * 80)


observed_sha256 = (
    r3_cell7_sha256(
        R3_CANDIDATE_FREEZE_PATH
    )
)


assert (
    observed_sha256
    ==
    EXPECTED_SHA256
), (
    "R3 candidate SHA256 mismatch.\n"
    f"Expected: {EXPECTED_SHA256}\n"
    f"Observed: {observed_sha256}"
)


print(
    "Expected SHA256:",
    EXPECTED_SHA256,
)

print(
    "Observed SHA256:",
    observed_sha256,
)

print(
    "SHA256 match:",
    True,
)


# ==============================================================================
# 6. RELOAD FROZEN PARQUET
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN CANDIDATE RELOAD")
print("=" * 80)


r3_frozen_reload = pd.read_parquet(
    R3_CANDIDATE_FREEZE_PATH,
    engine="pyarrow",
)


print(
    "Reloaded rows:",
    f"{len(r3_frozen_reload):,}",
)

print(
    "Reloaded columns:",
    len(
        r3_frozen_reload.columns
    ),
)


# ==============================================================================
# 7. SCHEMA VERIFICATION
# ==============================================================================

assert (
    list(
        r3_frozen_reload.columns
    )
    ==
    list(
        EXPECTED_COLUMNS
    )
), (
    "Frozen R3 schema mismatch."
)


print("\n" + "=" * 80)
print("SCHEMA VERIFICATION")
print("=" * 80)

print(
    "Column count:",
    len(
        r3_frozen_reload.columns
    ),
)

print(
    "Schema contract:",
    "PASS",
)


# ==============================================================================
# 8. POPULATION VERIFICATION
# ==============================================================================

observed_rows = len(
    r3_frozen_reload
)

observed_responses = (
    r3_frozen_reload[
        "response_id"
    ].nunique()
)

observed_sessions = (
    r3_frozen_reload[
        "session_id"
    ].nunique()
)

observed_objectives = (
    r3_frozen_reload[
        "objective_uid"
    ].nunique()
)

observed_folds = sorted(
    r3_frozen_reload[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert (
    observed_rows
    ==
    EXPECTED_ROWS
)

assert (
    observed_responses
    ==
    EXPECTED_RESPONSES
)

assert (
    observed_sessions
    ==
    EXPECTED_SESSIONS
)

assert (
    observed_objectives
    ==
    EXPECTED_OBJECTIVES
)

assert (
    observed_folds
    ==
    EXPECTED_FOLDS
)


print("\n" + "=" * 80)
print("POPULATION VERIFICATION")
print("=" * 80)

print(
    "Rows:",
    f"{observed_rows:,}",
)

print(
    "Responses:",
    f"{observed_responses:,}",
)

print(
    "Sessions:",
    f"{observed_sessions:,}",
)

print(
    "Objectives:",
    f"{observed_objectives:,}",
)

print(
    "Folds:",
    observed_folds,
)

print(
    "Population contract:",
    "PASS",
)


# ==============================================================================
# 9. IDENTITY VERIFICATION
# ==============================================================================

duplicate_identity_count = int(
    r3_frozen_reload.duplicated(
        subset=EXPECTED_IDENTITY_COLUMNS
    ).sum()
)


assert (
    duplicate_identity_count
    ==
    0
), (
    "Duplicate frozen candidate identities detected: "
    f"{duplicate_identity_count:,}"
)


print("\n" + "=" * 80)
print("IDENTITY VERIFICATION")
print("=" * 80)

print(
    "Identity columns:",
    EXPECTED_IDENTITY_COLUMNS,
)

print(
    "Duplicate identities:",
    duplicate_identity_count,
)

print(
    "Identity contract:",
    "PASS",
)


# ==============================================================================
# 10. PROVENANCE VERIFICATION
# ==============================================================================

assert (
    "selected_sparse"
    in
    r3_frozen_reload.columns
)

assert (
    "selected_dense"
    in
    r3_frozen_reload.columns
)


frozen_sparse_mask = (
    r3_frozen_reload[
        "selected_sparse"
    ]
    .fillna(False)
    .astype(bool)
)

frozen_dense_mask = (
    r3_frozen_reload[
        "selected_dense"
    ]
    .fillna(False)
    .astype(bool)
)


observed_sparse_only = int(
    (
        frozen_sparse_mask
        &
        ~frozen_dense_mask
    ).sum()
)

observed_dense_only = int(
    (
        ~frozen_sparse_mask
        &
        frozen_dense_mask
    ).sum()
)

observed_both = int(
    (
        frozen_sparse_mask
        &
        frozen_dense_mask
    ).sum()
)


assert (
    observed_sparse_only
    ==
    EXPECTED_SPARSE_ONLY
)

assert (
    observed_dense_only
    ==
    EXPECTED_DENSE_ONLY
)

assert (
    observed_both
    ==
    EXPECTED_BOTH
)


assert (
    observed_sparse_only
    +
    observed_dense_only
    +
    observed_both
    ==
    observed_rows
)


print("\n" + "=" * 80)
print("PROVENANCE VERIFICATION")
print("=" * 80)

print(
    "Sparse-only:",
    f"{observed_sparse_only:,}",
)

print(
    "Dense-only:",
    f"{observed_dense_only:,}",
)

print(
    "Both:",
    f"{observed_both:,}",
)

print(
    "Provenance contract:",
    "PASS",
)


# ==============================================================================
# 11. TARGET / LABEL ISOLATION
# ==============================================================================

TARGET_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
}


target_columns_present = (
    TARGET_COLUMNS
    &
    set(
        r3_frozen_reload.columns
    )
)


assert not target_columns_present, (
    "Target/label leakage detected in frozen R3 artifact: "
    + repr(
        sorted(
            target_columns_present
        )
    )
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target columns present:",
    sorted(
        target_columns_present
    ),
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 12. SESSION / RESPONSE INTEGRITY
# ==============================================================================

response_session_counts = (
    r3_frozen_reload
    .groupby(
        "response_id",
        sort=False,
    )[
        "session_id"
    ]
    .nunique()
)


response_objective_counts = (
    r3_frozen_reload
    .groupby(
        "response_id",
        sort=False,
    )[
        "objective_uid"
    ]
    .nunique()
)


assert (
    response_session_counts
    .eq(1)
    .all()
), (
    "A response is associated with multiple sessions."
)


assert (
    response_objective_counts
    .eq(1)
    .all()
), (
    "A response is associated with multiple objectives."
)


print("\n" + "=" * 80)
print("RESPONSE / SESSION INTEGRITY")
print("=" * 80)

print(
    "Responses with multiple sessions:",
    int(
        (
            response_session_counts
            > 1
        ).sum()
    ),
)

print(
    "Responses with multiple objectives:",
    int(
        (
            response_objective_counts
            > 1
        ).sum()
    ),
)

print(
    "Response/session contract:",
    "PASS",
)


# ==============================================================================
# 13. SCORE VALIDITY — PROVENANCE-AWARE, NOT PROBABILITY-BOUNDED
# ==============================================================================

print("\n" + "=" * 80)
print("SCORE VALIDITY — PROVENANCE-AWARE")
print("=" * 80)


# ------------------------------------------------------------------------------
# Parse score columns numerically.
#
# IMPORTANT:
# sparse_score is a retrieval ranking score, NOT a probability.
# Therefore we DO NOT impose [0, 1].
#
# dense_score is cosine similarity and may legitimately live in [-1, 1].
# ------------------------------------------------------------------------------

sparse_values = pd.to_numeric(
    r3_frozen_reload[
        "sparse_score"
    ],
    errors="coerce",
)

dense_values = pd.to_numeric(
    r3_frozen_reload[
        "dense_score"
    ],
    errors="coerce",
)


# ==============================================================================
# SPARSE SCORE
# ==============================================================================

# Sparse-selected candidates must have a valid sparse score.
sparse_selected_values = (
    sparse_values[
        frozen_sparse_mask
    ]
)


assert (
    sparse_selected_values.notna().all()
), (
    "Missing sparse_score on a sparse-selected candidate."
)


assert (
    np.isfinite(
        sparse_selected_values.to_numpy()
    ).all()
), (
    "Non-finite sparse_score on a sparse-selected candidate."
)


# ==============================================================================
# DENSE SCORE
# ==============================================================================

# Dense-selected candidates must have a valid dense score.
dense_selected_values = (
    dense_values[
        frozen_dense_mask
    ]
)


assert (
    dense_selected_values.notna().all()
), (
    "Missing dense_score on a dense-selected candidate."
)


assert (
    np.isfinite(
        dense_selected_values.to_numpy()
    ).all()
), (
    "Non-finite dense_score on a dense-selected candidate."
)


# ==============================================================================
# BOTH-SOURCE CANDIDATES
# ==============================================================================

both_mask_reload = (
    frozen_sparse_mask
    &
    frozen_dense_mask
)


assert (
    sparse_values[
        both_mask_reload
    ]
    .notna()
    .all()
), (
    "Missing sparse_score on a both-source candidate."
)


assert (
    dense_values[
        both_mask_reload
    ]
    .notna()
    .all()
), (
    "Missing dense_score on a both-source candidate."
)


# ==============================================================================
# DENSE COSINE RANGE
# ==============================================================================

# Sentence-transformer cosine similarity is bounded by [-1, 1].
# Small numerical tolerance is allowed.

dense_selected_array = (
    dense_selected_values
    .to_numpy(
        dtype=np.float64
    )
)


assert (
    dense_selected_array
    >= -1.0001
).all(), (
    "Dense cosine score below expected range."
)


assert (
    dense_selected_array
    <= 1.0001
).all(), (
    "Dense cosine score above expected range."
)


# ==============================================================================
# SPARSE RANGE — DIAGNOSTIC ONLY
# ==============================================================================

sparse_selected_array = (
    sparse_selected_values
    .to_numpy(
        dtype=np.float64
    )
)


sparse_min = float(
    sparse_selected_values.min()
)

sparse_max = float(
    sparse_selected_values.max()
)

dense_min = float(
    dense_selected_values.min()
)

dense_max = float(
    dense_selected_values.max()
)


# Do NOT assert [0,1] for sparse_score.
#
# We only require:
#   1. finite
#   2. numeric
#
# Ranking semantics, not probability semantics, apply here.


print(
    "Sparse score range:",
    f"{sparse_min:.8f}",
    "→",
    f"{sparse_max:.8f}",
)

print(
    "Dense score range:",
    f"{dense_min:.8f}",
    "→",
    f"{dense_max:.8f}",
)


# ==============================================================================
# EXPECTED MISSINGNESS
# ==============================================================================

sparse_score_nan_count = int(
    sparse_values.isna().sum()
)

dense_score_nan_count = int(
    dense_values.isna().sum()
)


sparse_only_mask_reload = (
    frozen_sparse_mask
    &
    ~frozen_dense_mask
)


dense_only_mask_reload = (
    ~frozen_sparse_mask
    &
    frozen_dense_mask
)


# Sparse score may be missing ONLY on dense-only candidates.
unexpected_sparse_nan_mask = (
    sparse_values.isna()
    &
    ~dense_only_mask_reload
)


# Dense score may be missing ONLY on sparse-only candidates.
unexpected_dense_nan_mask = (
    dense_values.isna()
    &
    ~sparse_only_mask_reload
)


unexpected_sparse_nan_count = int(
    unexpected_sparse_nan_mask.sum()
)

unexpected_dense_nan_count = int(
    unexpected_dense_nan_mask.sum()
)


assert (
    unexpected_sparse_nan_count
    ==
    0
), (
    "Sparse score missing outside allowed dense-only rows: "
    f"{unexpected_sparse_nan_count:,}"
)


assert (
    unexpected_dense_nan_count
    ==
    0
), (
    "Dense score missing outside allowed sparse-only rows: "
    f"{unexpected_dense_nan_count:,}"
)


# ==============================================================================
# FINAL SCORE VALIDITY
# ==============================================================================

print("\n" + "=" * 80)
print("SCORE VALIDITY SUMMARY")
print("=" * 80)

print(
    "Sparse selected rows:",
    f"{int(frozen_sparse_mask.sum()):,}",
)

print(
    "Dense selected rows:",
    f"{int(frozen_dense_mask.sum()):,}",
)

print(
    "Sparse score NaNs:",
    f"{sparse_score_nan_count:,}",
)

print(
    "Dense score NaNs:",
    f"{dense_score_nan_count:,}",
)

print(
    "Unexpected sparse NaNs:",
    unexpected_sparse_nan_count,
)

print(
    "Unexpected dense NaNs:",
    unexpected_dense_nan_count,
)

print(
    "Sparse numeric/finite validity:",
    "PASS",
)

print(
    "Dense numeric/finite validity:",
    "PASS",
)

print(
    "Dense cosine range:",
    "PASS",
)

print(
    "Sparse score probability-range check:",
    "NOT APPLIED",
)

print(
    "Score validity:",
    "PASS",
)

# ==============================================================================
# 14. FROZEN ARTIFACT RE-READ HASH
# ==============================================================================

# The parquet was read successfully after SHA256 validation.
# Recompute hash once more to ensure the artifact did not change during read.

post_read_sha256 = (
    r3_cell7_sha256(
        R3_CANDIDATE_FREEZE_PATH
    )
)


assert (
    post_read_sha256
    ==
    observed_sha256
)


print("\n" + "=" * 80)
print("POST-READ INTEGRITY")
print("=" * 80)

print(
    "Pre-read SHA256:",
    observed_sha256,
)

print(
    "Post-read SHA256:",
    post_read_sha256,
)

print(
    "Artifact unchanged:",
    True,
)


# ==============================================================================
# 15. CELL 7 READY
# ==============================================================================

R3_FROZEN_RELOAD_READY = True
R3_CELL_7_READY = True


print("\n" + "=" * 80)
print("R3 CELL 7 STATUS")
print("=" * 80)

print(
    "Frozen candidate exists:",
    True,
)

print(
    "Manifest valid:",
    True,
)

print(
    "SHA256 verified:",
    True,
)

print(
    "Population verified:",
    True,
)

print(
    "Identity verified:",
    True,
)

print(
    "Provenance verified:",
    True,
)

print(
    "Target isolation:",
    True,
)

print(
    "Score validity:",
    True,
)

print(
    "R3_FROZEN_RELOAD_READY:",
    R3_FROZEN_RELOAD_READY,
)

print(
    "R3_CELL_7_READY:",
    R3_CELL_7_READY,
)


assert (
    R3_FROZEN_RELOAD_READY
    is True
)

assert (
    R3_CELL_7_READY
    is True
)


print("=" * 80)
print(
    "R3 CELL 7 — FROZEN RELOAD VERIFICATION: PASS"
)
print("=" * 80)


# ==============================================================================
# 16. MEMORY CLEANUP
# ==============================================================================

if "r3_frozen_reload" in globals():
    del r3_frozen_reload

if "response_session_counts" in globals():
    del response_session_counts

if "response_objective_counts" in globals():
    del response_objective_counts

if "frozen_sparse_mask" in globals():
    del frozen_sparse_mask

if "frozen_dense_mask" in globals():
    del frozen_dense_mask

gc.collect()

print(
    "\nR3 Cell 7 memory cleanup: PASS"
)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION

FROZEN ARTIFACT PATHS
Frozen root: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen
Candidates: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_candidate_union.parquet
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_cell6_freeze_manifest.json

ARTIFACT EXISTENCE
Frozen directory: True
Candidate parquet: True
Freeze manifest: True

FREEZE MANIFEST
JSON valid: True
Status: FROZEN
Artifact: r3_candidate_union

SHA256 INTEGRITY
Expected SHA256: 4315ef4a5ecc7b99d6e2fbf341724861247baecf456dfa3cf38fae36aec982b7
Observed SHA256: 4315ef4a5ecc7b99d6e2fbf341724861247baecf456dfa3cf38fae36aec982b7
SHA256 match: True

FROZEN CANDIDATE RELOAD
Reloaded rows: 2,482,137
Reloaded columns: 20

SCHEMA VERIFICATION
Column count: 20
Schema contract: PASS

POPULATION VERIFICATION
Rows: 2,482

In [18]:
# ==============================================================================
# TRACE THE ACE — R3 CANDIDATE UNION
# CELL 8 — FINAL AUDIT / FREEZE GATE
# ==============================================================================

import gc
import hashlib
import json
from pathlib import Path

import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R3 CANDIDATE UNION")
print("CELL 8 — FINAL AUDIT / FREEZE GATE")
print("=" * 80)


# ==============================================================================
# 1. FROZEN PATHS
# ==============================================================================

R3_FREEZE_ROOT = (
    R3_ROOT
    / "frozen"
)

R3_CANDIDATE_FREEZE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

R3_FREEZE_MANIFEST_PATH = (
    R3_FREEZE_ROOT
    / "r3_cell6_freeze_manifest.json"
)


# ==============================================================================
# 2. UPSTREAM FROZEN ARTIFACTS
# ==============================================================================

R1_FROZEN_CANDIDATES = (
    R1_ROOT
    / "frozen"
    / "r1_sparse_candidates.parquet"
)

R1_FREEZE_MANIFEST = (
    R1_ROOT
    / "frozen"
    / "r1_cell6_freeze_manifest.json"
)

R2_FROZEN_CANDIDATES = (
    R2_ROOT
    / "frozen"
    / "r2_dense_candidates.parquet"
)

R2_FREEZE_MANIFEST = (
    R2_ROOT
    / "frozen"
    / "r2_dense_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("FROZEN ARTIFACT EXISTENCE")
print("=" * 80)


assert R3_FREEZE_ROOT.exists(), (
    f"Missing R3 frozen root:\n{R3_FREEZE_ROOT}"
)

assert R3_CANDIDATE_FREEZE_PATH.exists(), (
    f"Missing R3 frozen candidate:\n"
    f"{R3_CANDIDATE_FREEZE_PATH}"
)

assert R3_FREEZE_MANIFEST_PATH.exists(), (
    f"Missing R3 freeze manifest:\n"
    f"{R3_FREEZE_MANIFEST_PATH}"
)

assert R1_FROZEN_CANDIDATES.exists(), (
    f"Missing frozen R1 candidates:\n"
    f"{R1_FROZEN_CANDIDATES}"
)

assert R1_FREEZE_MANIFEST.exists(), (
    f"Missing R1 freeze manifest:\n"
    f"{R1_FREEZE_MANIFEST}"
)

assert R2_FROZEN_CANDIDATES.exists(), (
    f"Missing frozen R2 candidates:\n"
    f"{R2_FROZEN_CANDIDATES}"
)

assert R2_FREEZE_MANIFEST.exists(), (
    f"Missing R2 freeze manifest:\n"
    f"{R2_FREEZE_MANIFEST}"
)


print("R3 candidate parquet : True")
print("R3 freeze manifest    : True")
print("R1 frozen candidate   : True")
print("R1 freeze manifest    : True")
print("R2 frozen candidate   : True")
print("R2 freeze manifest    : True")


# ==============================================================================
# 3. HASH HELPER
# ==============================================================================

def r3_cell8_sha256(
    path,
    chunk_size=1024 * 1024,
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


# ==============================================================================
# 4. LOAD R3 MANIFEST
# ==============================================================================

with open(
    R3_FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    r3_manifest = json.load(
        handle
    )


assert isinstance(
    r3_manifest,
    dict
)

assert (
    r3_manifest.get("status")
    ==
    "FROZEN"
)

assert (
    r3_manifest.get("artifact")
    ==
    "r3_candidate_union"
)


print("\n" + "=" * 80)
print("R3 FREEZE MANIFEST")
print("=" * 80)

print(
    "JSON valid:",
    True,
)

print(
    "Status:",
    r3_manifest["status"],
)

print(
    "Artifact:",
    r3_manifest["artifact"],
)


# ==============================================================================
# 5. LOAD UPSTREAM MANIFESTS
# ==============================================================================

with open(
    R1_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    r1_manifest = json.load(
        handle
    )


with open(
    R2_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    r2_manifest = json.load(
        handle
    )


assert (
    r1_manifest.get("status")
    ==
    "FROZEN"
)

assert (
    r2_manifest.get("status")
    ==
    "FROZEN"
)


print(
    "R1 upstream manifest:",
    "FROZEN",
)

print(
    "R2 upstream manifest:",
    "FROZEN",
)


# ==============================================================================
# 6. R3 POPULATION CONTRACT
# ==============================================================================

EXPECTED_R3_ROWS = int(
    r3_manifest["rows"]
)

EXPECTED_R3_RESPONSES = int(
    r3_manifest["responses"]
)

EXPECTED_R3_SESSIONS = int(
    r3_manifest["sessions"]
)

EXPECTED_R3_OBJECTIVES = int(
    r3_manifest["objectives"]
)

EXPECTED_R3_FOLDS = sorted(
    [
        int(x)
        for x in
        r3_manifest["folds"]
    ]
)


assert (
    EXPECTED_R3_ROWS
    ==
    2_482_137
)

assert (
    EXPECTED_R3_RESPONSES
    ==
    35_072
)

assert (
    EXPECTED_R3_SESSIONS
    ==
    22_821
)

assert (
    EXPECTED_R3_OBJECTIVES
    ==
    398
)

assert (
    EXPECTED_R3_FOLDS
    ==
    [0, 1, 2, 3, 4]
)


print("\n" + "=" * 80)
print("R3 POPULATION CONTRACT")
print("=" * 80)

print(
    "Rows:",
    f"{EXPECTED_R3_ROWS:,}",
)

print(
    "Responses:",
    f"{EXPECTED_R3_RESPONSES:,}",
)

print(
    "Sessions:",
    f"{EXPECTED_R3_SESSIONS:,}",
)

print(
    "Objectives:",
    f"{EXPECTED_R3_OBJECTIVES:,}",
)

print(
    "Folds:",
    EXPECTED_R3_FOLDS,
)

print(
    "Population contract: PASS"
)


# ==============================================================================
# 7. UPSTREAM POPULATION CONTRACT
# ==============================================================================

assert (
    int(
        r1_manifest.get(
            "rows",
            0,
        )
    )
    ==
    1_752_048
)

assert (
    int(
        r2_manifest.get(
            "rows",
            0,
        )
    )
    ==
    1_752_048
)


print("\n" + "=" * 80)
print("UPSTREAM POPULATION CONTRACT")
print("=" * 80)

print(
    "R1 frozen rows:",
    f"{int(r1_manifest['rows']):,}",
)

print(
    "R2 frozen rows:",
    f"{int(r2_manifest['rows']):,}",
)

print(
    "R1/R2 population contract: PASS"
)


# ==============================================================================
# 8. RELOAD FINAL R3 ARTIFACT
# ==============================================================================

print("\n" + "=" * 80)
print("FINAL FROZEN ARTIFACT RELOAD")
print("=" * 80)


r3_final = pd.read_parquet(
    R3_CANDIDATE_FREEZE_PATH,
    engine="pyarrow",
)


assert len(
    r3_final
) == EXPECTED_R3_ROWS


print(
    "Reloaded rows:",
    f"{len(r3_final):,}",
)


# ==============================================================================
# 9. SCHEMA CONTRACT
# ==============================================================================

expected_columns = list(
    r3_manifest[
        "schema_columns"
    ]
)

identity_columns = list(
    r3_manifest[
        "identity_columns"
    ]
)


assert (
    list(
        r3_final.columns
    )
    ==
    expected_columns
)


for column in identity_columns:

    assert (
        column
        in
        r3_final.columns
    )


print("\n" + "=" * 80)
print("SCHEMA CONTRACT")
print("=" * 80)

print(
    "Columns:",
    len(
        r3_final.columns
    ),
)

print(
    "Identity columns:",
    identity_columns,
)

print(
    "Schema contract: PASS"
)


# ==============================================================================
# 10. FINAL IDENTITY AUDIT
# ==============================================================================

duplicate_identity_count = int(
    r3_final.duplicated(
        subset=identity_columns
    ).sum()
)


assert (
    duplicate_identity_count
    ==
    0
)


print("\n" + "=" * 80)
print("IDENTITY AUDIT")
print("=" * 80)

print(
    "Duplicate candidate identities:",
    duplicate_identity_count,
)

print(
    "Identity audit: PASS"
)


# ==============================================================================
# 11. RESPONSE / SESSION / OBJECTIVE / FOLD AUDIT
# ==============================================================================

observed_responses = (
    r3_final[
        "response_id"
    ].nunique()
)

observed_sessions = (
    r3_final[
        "session_id"
    ].nunique()
)

observed_objectives = (
    r3_final[
        "objective_uid"
    ].nunique()
)

observed_folds = sorted(
    r3_final[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert (
    observed_responses
    ==
    EXPECTED_R3_RESPONSES
)

assert (
    observed_sessions
    ==
    EXPECTED_R3_SESSIONS
)

assert (
    observed_objectives
    ==
    EXPECTED_R3_OBJECTIVES
)

assert (
    observed_folds
    ==
    EXPECTED_R3_FOLDS
)


print("\n" + "=" * 80)
print("COVERAGE AUDIT")
print("=" * 80)

print(
    "Responses:",
    f"{observed_responses:,}",
)

print(
    "Sessions:",
    f"{observed_sessions:,}",
)

print(
    "Objectives:",
    f"{observed_objectives:,}",
)

print(
    "Folds:",
    observed_folds,
)

print(
    "Coverage audit: PASS"
)


# ==============================================================================
# 12. PROVENANCE AUDIT
# ==============================================================================

sparse_mask_final = (
    r3_final[
        "selected_sparse"
    ]
    .fillna(False)
    .astype(bool)
)

dense_mask_final = (
    r3_final[
        "selected_dense"
    ]
    .fillna(False)
    .astype(bool)
)


sparse_only_final = int(
    (
        sparse_mask_final
        &
        ~dense_mask_final
    ).sum()
)

dense_only_final = int(
    (
        ~sparse_mask_final
        &
        dense_mask_final
    ).sum()
)

both_final = int(
    (
        sparse_mask_final
        &
        dense_mask_final
    ).sum()
)


manifest_provenance = (
    r3_manifest[
        "provenance"
    ]
)


assert (
    sparse_only_final
    ==
    int(
        manifest_provenance[
            "sparse_only"
        ]
    )
)

assert (
    dense_only_final
    ==
    int(
        manifest_provenance[
            "dense_only"
        ]
    )
)

assert (
    both_final
    ==
    int(
        manifest_provenance[
            "both_sources"
        ]
    )
)


assert (
    sparse_only_final
    +
    dense_only_final
    +
    both_final
    ==
    EXPECTED_R3_ROWS
)


print("\n" + "=" * 80)
print("PROVENANCE AUDIT")
print("=" * 80)

print(
    "Sparse-only:",
    f"{sparse_only_final:,}",
)

print(
    "Dense-only:",
    f"{dense_only_final:,}",
)

print(
    "Both:",
    f"{both_final:,}",
)

print(
    "Provenance accounting: PASS"
)


# ==============================================================================
# 13. TARGET ISOLATION
# ==============================================================================

TARGET_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
}


target_columns_found = (
    TARGET_COLUMNS
    &
    set(
        r3_final.columns
    )
)


assert not target_columns_found


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target columns found:",
    sorted(
        target_columns_found
    ),
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation: PASS"
)


# ==============================================================================
# 14. RESPONSE SESSION CONSISTENCY
# ==============================================================================

response_session_counts = (
    r3_final
    .groupby(
        "response_id",
        sort=False,
    )[
        "session_id"
    ]
    .nunique()
)


response_objective_counts = (
    r3_final
    .groupby(
        "response_id",
        sort=False,
    )[
        "objective_uid"
    ]
    .nunique()
)


assert (
    response_session_counts
    .eq(1)
    .all()
)

assert (
    response_objective_counts
    .eq(1)
    .all()
)


print("\n" + "=" * 80)
print("RESPONSE CONSISTENCY")
print("=" * 80)

print(
    "Responses spanning multiple sessions:",
    int(
        (
            response_session_counts
            > 1
        ).sum()
    ),
)

print(
    "Responses spanning multiple objectives:",
    int(
        (
            response_objective_counts
            > 1
        ).sum()
    ),
)

print(
    "Response consistency: PASS"
)


# ==============================================================================
# 15. REQUIRED R1/R2 PROVENANCE CONTRACT
# ==============================================================================

assert (
    "selected_sparse"
    in
    r3_final.columns
)

assert (
    "selected_dense"
    in
    r3_final.columns
)

assert (
    "sparse_score"
    in
    r3_final.columns
)

assert (
    "dense_score"
    in
    r3_final.columns
)


# A candidate must originate from at least one retrieval source.

assert (
    (
        sparse_mask_final
        |
        dense_mask_final
    )
    .all()
)


print("\n" + "=" * 80)
print("R1/R2 PROVENANCE CONTRACT")
print("=" * 80)

print(
    "Every candidate has sparse or dense provenance:",
    True,
)

print(
    "R1/R2 provenance contract: PASS"
)


# ==============================================================================
# 16. HASH VERIFICATION
# ==============================================================================

expected_sha256 = (
    r3_manifest[
        "candidate_sha256"
    ]
)


observed_sha256 = (
    r3_cell8_sha256(
        R3_CANDIDATE_FREEZE_PATH
    )
)


assert (
    observed_sha256
    ==
    expected_sha256
)


print("\n" + "=" * 80)
print("FINAL ARTIFACT HASH")
print("=" * 80)

print(
    "Manifest SHA256:",
    expected_sha256,
)

print(
    "Observed SHA256:",
    observed_sha256,
)

print(
    "SHA256 verification: PASS"
)


# ==============================================================================
# 17. MANIFEST ↔ ARTIFACT CONTRACT
# ==============================================================================

assert (
    r3_manifest[
        "rows"
    ]
    ==
    len(
        r3_final
    )
)

assert (
    r3_manifest[
        "responses"
    ]
    ==
    observed_responses
)

assert (
    r3_manifest[
        "sessions"
    ]
    ==
    observed_sessions
)

assert (
    r3_manifest[
        "objectives"
    ]
    ==
    observed_objectives
)

assert (
    sorted(
        r3_manifest[
            "folds"
        ]
    )
    ==
    observed_folds
)


print("\n" + "=" * 80)
print("MANIFEST ↔ ARTIFACT CONTRACT")
print("=" * 80)

print(
    "Manifest and artifact population:",
    "MATCH",
)

print(
    "Manifest and artifact schema:",
    "MATCH",
)

print(
    "Manifest and artifact provenance:",
    "MATCH",
)

print(
    "Manifest ↔ artifact contract: PASS"
)


# ==============================================================================
# 18. FINAL FREEZE GATE
# ==============================================================================

R3_FINAL_FREEZE_GATE = True
R3_CELL_8_READY = True
R3_RETRIEVAL_FROZEN = True


print("\n" + "=" * 80)
print("R3 FINAL FREEZE GATE")
print("=" * 80)

print(
    "R1 frozen dependency:",
    True,
)

print(
    "R2 frozen dependency:",
    True,
)

print(
    "R3 candidate union frozen:",
    True,
)

print(
    "R3 frozen reload verified:",
    True,
)

print(
    "Schema verified:",
    True,
)

print(
    "Population verified:",
    True,
)

print(
    "Identity verified:",
    True,
)

print(
    "Coverage verified:",
    True,
)

print(
    "Provenance verified:",
    True,
)

print(
    "Target isolation verified:",
    True,
)

print(
    "SHA256 verified:",
    True,
)

print(
    "Manifest/artifact consistency:",
    True,
)


assert (
    R3_FINAL_FREEZE_GATE
    is True
)

assert (
    R3_CELL_8_READY
    is True
)

assert (
    R3_RETRIEVAL_FROZEN
    is True
)


print("\n" + "=" * 80)
print("R3 CELL 8 STATUS")
print("=" * 80)

print(
    "R3_FINAL_FREEZE_GATE:",
    R3_FINAL_FREEZE_GATE,
)

print(
    "R3_CELL_8_READY:",
    R3_CELL_8_READY,
)

print(
    "R3_RETRIEVAL_FROZEN:",
    R3_RETRIEVAL_FROZEN,
)

print("=" * 80)
print(
    "R3 CELL 8 — FINAL AUDIT / FREEZE GATE: PASS"
)
print("=" * 80)


# ==============================================================================
# 19. MEMORY CLEANUP
# ==============================================================================

if "r3_final" in globals():
    del r3_final

if "response_session_counts" in globals():
    del response_session_counts

if "response_objective_counts" in globals():
    del response_objective_counts

if "sparse_mask_final" in globals():
    del sparse_mask_final

if "dense_mask_final" in globals():
    del dense_mask_final

gc.collect()

print(
    "\nR3 Cell 8 memory cleanup: PASS"
)

TRACE THE ACE — R3 CANDIDATE UNION
CELL 8 — FINAL AUDIT / FREEZE GATE

FROZEN ARTIFACT EXISTENCE
R3 candidate parquet : True
R3 freeze manifest    : True
R1 frozen candidate   : True
R1 freeze manifest    : True
R2 frozen candidate   : True
R2 freeze manifest    : True

R3 FREEZE MANIFEST
JSON valid: True
Status: FROZEN
Artifact: r3_candidate_union
R1 upstream manifest: FROZEN
R2 upstream manifest: FROZEN

R3 POPULATION CONTRACT
Rows: 2,482,137
Responses: 35,072
Sessions: 22,821
Objectives: 398
Folds: [0, 1, 2, 3, 4]
Population contract: PASS

UPSTREAM POPULATION CONTRACT
R1 frozen rows: 1,752,048
R2 frozen rows: 1,752,048
R1/R2 population contract: PASS

FINAL FROZEN ARTIFACT RELOAD
Reloaded rows: 2,482,137

SCHEMA CONTRACT
Columns: 20
Identity columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid']
Schema contract: PASS

IDENTITY AUDIT
Duplicate candidate identities: 0
Identity audit: PASS

COVERAGE AUDIT
Responses: 35,072
Sessions: 22,821
Objectives: 398
Folds: [